# Version 10: Plate-Preserving Dish-to-3D Demo & Docker Worker Blueprint

Private ticket-ready demo (`86bd5m6q0`, `publication_eligible: False`) and reproducible **Docker worker blueprint**
supporting **user-uploaded dish photos** (Stage 0 private dataset, up to 4 dishes) or synthetic dish inputs on Kaggle
CUDA GPUs (`T4 ×2` or single GPU):

1. **Plate-Preserving Dish Input & Foreground Cutout (`Stage 0/1`):**
   - **Stage 0 (`user-uploaded-dataset`):** Discovers up to 4 user photos under `/kaggle` `input`, computes exact
     `source_sha256`, and binds optional sibling `<stem>_mask.<ext>` full-plate masks.
   - **Plate-Preserving Segmentation (`preserve_plate`):** Extracts the food matte via `birefnet-general-lite`
     (`isnet-general-use` → `u2net` fallback), then detects and segments the supporting plate/bowl/tray/board using
     pinned Apache-2.0 **`IDEA-Research/grounding-dino-tiny` + `facebook/sam-vit-base`** (plus ring-point prompts and
     corner tabletop-leak guards) so the **entire dish + plate** stays intact for 3D reconstruction.

2. **3D Reconstruction Stage (Raw + Mobile-Optimized `<=35k` GLB Export Profiles):**
   - **`TripoSR`** (`resolution=256`, glTF `+Y` up / `+Z` front rotation `[:, [1, 2, 0]]`, floater pruning) →
     `triposr.glb` + decimated **`triposr_mobile.glb`** (`<=35k` faces, vertex colors preserved).
   - **`Hunyuan3D-2mini`** (`8%` square margin padding, `mc_algo="mc"`, `FloaterRemover` + `DegenerateFaceRemover` +
     `FaceReducer`) → `hunyuan3d_2mini.glb` (`<=80k` faces) + **`hunyuan3d_2mini_mobile.glb`** (`<=35k` faces).
   - **`TRELLIS.2-4B`**: preflight-checked on gated DINOv3 before compilation.

3. **HF TRELLIS Demo-Inspired Studio Inspection & Docker Packaging Blueprint:**
   - Software-renders `Normal`, `Clay`, `Base Color`, and `Studio HDRI` (`hf_modes_sheet.png`), sRGB-corrected
     3-angle strips (`comparison_sheet.png`), and an interactive `<model-viewer>` studio (`comparison_viewer.html`) with
     dish tabs, raw-vs-mobile toggles, exposure/auto-rotate controls, and truthful private single-photo demo labeling.
   - Emits per-dish immutable `job_manifest.json` → `worker_output.json` receipts and a runnable **`docker_blueprint/`**
     (`Dockerfile.worker.template`, `worker_entrypoint.py`, `requirements.lock.txt`, `pinned_revisions.json`).


In [ ]:
import base64
import binascii
import hashlib
import json
import re
import struct
import time
from io import BytesIO
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import requests
from PIL import Image, ImageOps

CHAT_API = 'https://openrouter.ai/api/v1/chat/completions'
OPENROUTER_MODELS = {
    'gemini-3.1-flash-lite-image': 'google/gemini-3.1-flash-lite-image',
    'gemini-3.1-flash-image': 'google/gemini-3.1-flash-image',
}
FALLBACK_IMAGE_MODELS = {
    'sd-turbo': {
        'repo': 'stabilityai/sd-turbo',
        'revision': 'b261bac6fd2cf515557d5d0707481eafa0485ec2',
        'steps': 4,
        'guidance': 0.0,
        'size': 512,
        'seed': 42,
    },
    'dreamshaper-8': {
        'repo': 'Lykon/dreamshaper-8',
        'revision': 'a7e52b98680b1ba8ff7bce97c7f9f2e2e5337917',
        'steps': 25,
        'guidance': 7.0,
        'size': 512,
        'seed': 42,
    },
}
DINOV3_REPO = 'facebook/dinov3-vitl16-pretrain-lvd1689m'
DINOV3_REVISION = 'ea8dc2863c51be0a264bab82070e3e8836b02d51'
TRELLIS_MODEL_REPO = 'microsoft/TRELLIS.2-4B'
TRELLIS_MODEL_REVISION = 'af44b45f2e35a493886929c6d786e563ec68364d'
GROUNDING_DINO_REPO = 'IDEA-Research/grounding-dino-tiny'
GROUNDING_DINO_REVISION = 'a2bb814dd30d776dcf7e30523b00659f4f141c71'
SAM_REPO = 'facebook/sam-vit-base'
SAM_REVISION = '70c1a07f894ebb5b307fd9eaaee97b9dfc16068f'
PLATE_PRESERVATION_MODE = 'preserve_plate'
MOBILE_OPTIMIZED_MAX_FACES = 35000
DEMO_TRUTHFUL_LABEL = 'Private single-photo 3D demo (not published T0/T1/T2 menu evidence)'
DISH_PROMPT = (
    'Studio food photo of fictional dish Ember Garden: five browned purple sweet potato dumplings, '
    'roasted golden carrots, and parsley on one matte ivory round plate. '
    'Elevated three-quarter view showing the full plate rim with wide margin on a plain dark slate background, '
    'soft diffuse light. Only one plate; no utensils, bowls, hands, or text.'
)
MAX_RESPONSE_BYTES = 32 * 1024 * 1024
MAX_IMAGE_BYTES = 20 * 1024 * 1024
MAX_PIXELS = 4096 * 4096
MIME_TO_FORMAT = {'image/png': 'PNG', 'image/jpeg': 'JPEG', 'image/webp': 'WEBP'}
DATA_URL_RE = re.compile(r'^data:(image/(?:png|jpeg|webp));base64,(.+)$', re.DOTALL)
MAX_USER_DISHES = 4
SUPPORTED_IMAGE_SUFFIXES = ('.png', '.jpg', '.jpeg', '.webp')


class DemoError(RuntimeError):
    """A fixed diagnostic safe to display or package without leaking secrets."""


def kaggle_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None


def preflight_dinov3(token, downloader=None):
    """Check gated DINOv3 access via small config.json before heavy builds or paid calls."""
    try:
        if downloader is None:
            from huggingface_hub import hf_hub_download
            downloader = hf_hub_download
        downloader(DINOV3_REPO, 'config.json', revision=DINOV3_REVISION, token=token or False)
        return True, 'ok'
    except Exception as exc:
        name = type(exc).__name__
        message = str(exc)
        if any(code in message for code in ('401', '403', 'GatedRepo')) or 'gated' in message.lower():
            return False, 'missing_or_unauthorized_hf_token'
        return False, f'preflight_unavailable:{name}'


def decode_image(payload):
    try:
        if not isinstance(payload, dict) or payload.get('error'):
            raise ValueError
        choices = payload['choices']
        if not isinstance(choices, list) or not choices:
            raise ValueError
        images = choices[0]['message']['images']
        if not isinstance(images, list) or len(images) != 1:
            raise ValueError
        url = images[0]['image_url']['url']
        if not isinstance(url, str):
            raise ValueError
        match = DATA_URL_RE.match(url.strip())
        if not match:
            raise ValueError
        declared_mime, encoded = match.group(1), match.group(2)
        if len(encoded) > 4 * ((MAX_IMAGE_BYTES + 2) // 3):
            raise ValueError
        raw = base64.b64decode(encoded, validate=True)
        if not raw or len(raw) > MAX_IMAGE_BYTES:
            raise ValueError
        with Image.open(BytesIO(raw)) as image:
            width, height = image.size
            if (image.format != MIME_TO_FORMAT[declared_mime]
                    or width * height > MAX_PIXELS
                    or getattr(image, 'n_frames', 1) != 1):
                raise ValueError
            image.verify()
        with Image.open(BytesIO(raw)) as image:
            mode = 'RGBA' if 'A' in image.getbands() or 'transparency' in image.info else 'RGB'
            pixels = ImageOps.exif_transpose(image).convert(mode)
            pixels.info.clear()
            return pixels
    except (AttributeError, binascii.Error, Image.DecompressionBombError, KeyError,
            OSError, SyntaxError, TypeError, ValueError):
        raise DemoError('OpenRouter returned an invalid or unsupported image.') from None


def request_image(model, api_key, post=requests.post, sleep=time.sleep, max_retries=2):
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            with post(
                CHAT_API,
                headers={'Authorization': f'Bearer {api_key}'},
                json={
                    'model': model,
                    'messages': [{'role': 'user', 'content': DISH_PROMPT}],
                    'modalities': ['image', 'text'],
                    'image_config': {'aspect_ratio': '1:1'},
                },
                timeout=(15, 180),
                stream=True,
                allow_redirects=False,
            ) as response:
                if response.status_code == 429:
                    last_error = DemoError('OpenRouter HTTP 429.')
                    if attempt < max_retries:
                        sleep(2 * (attempt + 1))
                        continue
                    raise last_error
                if response.status_code != 200:
                    raise DemoError(f'OpenRouter HTTP {response.status_code}.')
                body = bytearray()
                for chunk in response.iter_content(64 * 1024):
                    body.extend(chunk)
                    if len(body) > MAX_RESPONSE_BYTES:
                        raise DemoError('OpenRouter response exceeded the size limit.')
            return decode_image(json.loads(body))
        except DemoError:
            raise
        except requests.RequestException:
            raise DemoError('OpenRouter connection or read failed.') from None
        except (UnicodeError, ValueError):
            raise DemoError('OpenRouter returned invalid JSON.') from None
    raise last_error or DemoError('OpenRouter request failed.')


def discover_user_dish_images(results_dir, input_root=None, max_dishes=MAX_USER_DISHES):
    """Discover up to max_dishes attached user dish photos and optional sibling _mask files."""
    root = Path(input_root) if input_root is not None else Path('/kaggle') / 'input'
    if not root.is_dir():
        return {}
    all_images = sorted(
        (
            path for path in root.rglob('*')
            if path.is_file()
            and not path.name.startswith('.')
            and path.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES
        ),
        key=lambda path: path.relative_to(root).as_posix(),
    )
    candidates = [p for p in all_images if not p.stem.endswith('_mask')][:max_dishes]
    mask_lookup = {
        (p.parent, p.stem[:-5]): p
        for p in all_images
        if p.stem.endswith('_mask')
    }
    results_dir = Path(results_dir)
    discovered = {}
    for idx, src in enumerate(candidates, start=1):
        raw_label = re.sub(r'[^a-zA-Z0-9._-]+', '-', src.stem).strip('-.') or f'dish_{idx}'
        label = raw_label if raw_label not in discovered else f'{raw_label}_{idx}'
        folder = results_dir / label
        folder.mkdir(parents=True, exist_ok=True)
        target = folder / 'input_image.png'
        reviewed_mask = mask_lookup.get((src.parent, src.stem))
        try:
            with Image.open(src) as im:
                width, height = im.size
                if getattr(im, 'n_frames', 1) != 1 or width * height > MAX_PIXELS:
                    raise ValueError('Unsupported multi-frame or oversized input image.')
                mode = 'RGBA' if 'A' in im.getbands() or 'transparency' in im.info else 'RGB'
                pixels = ImageOps.exif_transpose(im).convert(mode)
                pixels.info.clear()
                pixels.save(target)
                size = list(pixels.size)
            with target.open('rb') as stream:
                source_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
            discovered[label] = {
                'label': label,
                'model': 'user-photo-upload',
                'image_source': 'user-uploaded-dataset',
                'original_filename': src.name,
                'dataset_slug': src.parent.name,
                'size': size,
                'source_sha256': source_sha256,
                'image_status': 'success',
                'image_path': target,
                'reviewed_mask_path': reviewed_mask,
                'directory': folder,
            }
        except Exception as err:
            discovered[label] = {
                'label': label,
                'model': 'user-photo-upload',
                'image_source': 'user-uploaded-dataset',
                'original_filename': src.name,
                'image_status': 'failed',
                'image_error': f'Invalid user dish photo ({type(err).__name__}).',
                'directory': folder,
            }
    return discovered


def build_job_manifest(
    label,
    input_entry,
    revisions,
    plate_mode=PLATE_PRESERVATION_MODE,
    seed=42,
    mobile_max_faces=MOBILE_OPTIMIZED_MAX_FACES,
):
    """Build an immutable Path-free Docker worker input contract for one dish photo."""
    src_path = input_entry.get('image_path')
    src_sha = input_entry.get('source_sha256')
    if not src_sha and src_path and Path(src_path).is_file():
        with Path(src_path).open('rb') as stream:
            src_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
    seed_bytes = f'{label}:{src_sha or "none"}:{plate_mode}:{seed}'.encode('utf-8')
    job_id = 'demo-' + hashlib.sha256(seed_bytes).hexdigest()[:16]
    reviewed = input_entry.get('reviewed_mask_path')
    return {
        'schema_version': 1,
        'job_id': job_id,
        'dish_label': label,
        'purpose': 'private single-photo 3D demo',
        'demo_label': DEMO_TRUTHFUL_LABEL,
        'publication_eligible': False,
        'image_source': input_entry.get('image_source', 'unknown'),
        'original_filename': input_entry.get('original_filename', 'input_image.png'),
        'input_filename': Path(src_path).name if src_path else 'input_image.png',
        'input_size': list(input_entry.get('size', [])),
        'source_sha256': src_sha,
        'plate_preservation_mode': plate_mode,
        'reviewed_mask': Path(reviewed).name if reviewed else None,
        'seed': int(seed),
        'export_profiles': {
            'raw': {'triposr_resolution': 256, 'hunyuan_max_faces': 80000},
            'mobile': {'max_faces': int(mobile_max_faces), 'format': 'glb'},
        },
        'pinned_revisions': {
            k: v for k, v in revisions.items()
            if not k.startswith(('input_', 'cutout_'))
        },
    }


def build_worker_receipt(job_manifest, input_entry, dish_runs, results_dir):
    """Build a verifiable Path-free Docker worker output receipt from generated artifacts."""
    root = Path(results_dir)
    cut_path = input_entry.get('cutout_path')
    cut_rec = file_record(cut_path, root) if cut_path and Path(cut_path).is_file() else None
    models = {}
    for run in dish_runs.values():
        backend = run.get('backend', 'unknown')
        entry = {
            'status': run.get('status', 'unknown'),
            'appearance': run.get('appearance', 'unknown'),
        }
        if run.get('status') == 'success' and run.get('artifact_paths'):
            raw_glb = Path(run['artifact_paths'][0])
            raw_info = file_record(raw_glb, root)
            raw_m = run.get('metrics', {})
            entry['raw_glb'] = {
                **raw_info,
                'vertices': raw_m.get('vertices', raw_m.get('vertex_count')),
                'faces': raw_m.get('faces'),
                'has_vertex_color': raw_m.get('has_vertex_color'),
                'elapsed_seconds': raw_m.get('elapsed_seconds'),
            }
            mob_path = run.get('mobile_glb_path')
            if mob_path and Path(mob_path).is_file():
                mob_info = file_record(mob_path, root)
                mob_m = run.get('mobile_metrics', {})
                entry['mobile_glb'] = {
                    **mob_info,
                    'vertices': mob_m.get('vertices', mob_m.get('vertex_count')),
                    'faces': mob_m.get('faces'),
                    'has_vertex_color': mob_m.get('has_vertex_color'),
                }
        elif run.get('error'):
            entry['error'] = run['error']
        models[backend] = entry
    cut_metrics = dict(input_entry.get('cutout_metrics', {}))
    return {
        'schema_version': 1,
        'job_id': job_manifest['job_id'],
        'dish_label': job_manifest['dish_label'],
        'demo_label': DEMO_TRUTHFUL_LABEL,
        'publication_eligible': False,
        'source_sha256': job_manifest.get('source_sha256'),
        'cutout_sha256': cut_rec['sha256'] if cut_rec else None,
        'plate_preserved': bool(cut_metrics.get('plate_preserved', False)),
        'cutout_metrics': cut_metrics,
        'models': models,
        'pinned_revisions': job_manifest.get('pinned_revisions', {}),
    }

def collect_openrouter_images(directory, api_key, generate=request_image):
    attempts = {}
    for label, model in OPENROUTER_MODELS.items():
        folder = directory / label
        folder.mkdir(parents=True, exist_ok=True)
        entry = attempts[label] = {
            'label': label,
            'model': model,
            'image_source': 'openrouter',
            'directory': folder,
        }
        try:
            if not api_key:
                raise DemoError('OPENROUTER_API_KEY not attached in Kaggle Secrets.')
            with generate(model, api_key) as image:
                target = folder / 'input_image.png'
                image.save(target)
                entry['size'] = list(image.size)
            entry['image_status'] = 'success'
            entry['image_path'] = target
        except DemoError as error:
            entry.update(image_status='failed', image_error=str(error))
        except OSError:
            entry.update(image_status='failed', image_error='Could not save conditioning image.')
    return attempts


def validate_glb(path):
    path = Path(path)
    with path.open('rb') as stream:
        if stream.read(4) != b'glTF':
            raise DemoError('Invalid GLB magic.')
        header = stream.read(16)
        if len(header) != 16:
            raise DemoError('Truncated GLB header.')
        version, length, json_length, chunk_type = struct.unpack('<IIII', header)
        file_size = path.stat().st_size
        if (version != 2 or length != file_size or chunk_type != 0x4E4F534A
                or json_length == 0 or json_length % 4 or json_length > min(length - 20, 16 * 1024 * 1024)):
            raise DemoError('Invalid GLB version or chunk bounds.')
        try:
            data = json.loads(stream.read(json_length))
        except (UnicodeError, ValueError):
            raise DemoError('Invalid GLB JSON chunk.') from None
        try:
            meshes = data['meshes']
            accessors = data['accessors']
            primitives = [primitive for mesh in meshes for primitive in mesh['primitives']]
            if not primitives:
                raise ValueError
            vertex_count = 0
            has_vertex_color = False
            has_texcoord = False
            for primitive in primitives:
                attrs = primitive['attributes']
                pos_idx = attrs['POSITION']
                if (type(pos_idx) is not int or not 0 <= pos_idx < len(accessors)
                        or accessors[pos_idx]['type'] != 'VEC3' or accessors[pos_idx]['count'] <= 0):
                    raise ValueError
                vertex_count += int(accessors[pos_idx]['count'])
                if 'COLOR_0' in attrs:
                    has_vertex_color = True
                if 'TEXCOORD_0' in attrs:
                    has_texcoord = True
        except (IndexError, KeyError, TypeError, ValueError):
            raise DemoError('GLB contains no valid mesh position accessors.') from None
        remaining = length - stream.tell()
        if remaining:
            binary_header = stream.read(8)
            if len(binary_header) != 8:
                raise DemoError('Truncated GLB binary chunk header.')
            binary_length, binary_type = struct.unpack('<II', binary_header)
            if binary_type != 0x004E4942 or binary_length % 4 or binary_length != remaining - 8:
                raise DemoError('Invalid GLB binary chunk bounds.')
    return {
        'bytes': file_size,
        'meshes': len(meshes),
        'primitives': len(primitives),
        'vertex_count': vertex_count,
        'has_vertex_color': has_vertex_color,
        'has_texture': bool(data.get('textures')) and has_texcoord,
    }


def file_record(path, root):
    path = Path(path)
    with path.open('rb') as stream:
        digest = hashlib.file_digest(stream, 'sha256').hexdigest()
    return {'path': path.relative_to(root).as_posix(), 'sha256': digest, 'bytes': path.stat().st_size}


def write_results(directory, inputs, runs_3d, settings, revisions, extra_files=()):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    manifest = {
        'schema_version': 3,
        'purpose': 'private single-photo 3D demo & Docker worker blueprint',
        'demo_label': DEMO_TRUTHFUL_LABEL,
        'plate_preservation_mode': PLATE_PRESERVATION_MODE,
        'publication_eligible': False,
        'prompt': DISH_PROMPT,
        'trellis_settings': settings,
        'source_revisions': revisions,
        'inputs': {},
        'runs_3d': {},
        'extras': [],
    }
    archived = {}

    def stage(path):
        path = Path(path)
        rec = file_record(path, directory)
        archived[rec['path']] = path
        return rec

    for label, info in inputs.items():
        clean = {
            k: v for k, v in info.items()
            if k not in ('directory', 'image_path', 'cutout_path', 'reviewed_mask_path', 'job_manifest_path')
        }
        artifacts = []
        for key in ('image_path', 'cutout_path', 'job_manifest_path'):
            candidate = info.get(key)
            if candidate and Path(candidate).is_file():
                artifacts.append(stage(candidate))
        clean['artifacts'] = artifacts
        manifest['inputs'][label] = clean

    for key, run_info in runs_3d.items():
        clean = {
            k: v for k, v in run_info.items()
            if k not in ('directory', 'artifact_paths', 'preview_path', 'hf_modes_path', 'mobile_glb_path')
        }
        artifacts = []
        for path in run_info.get('artifact_paths', []):
            if Path(path).is_file():
                artifacts.append(stage(path))
        clean['artifacts'] = artifacts
        manifest['runs_3d'][key] = clean

    for extra in extra_files:
        if extra and Path(extra).is_file():
            manifest['extras'].append(stage(extra))

    manifest_text = json.dumps(manifest, indent=2) + '\n'
    with ZipFile(directory / 'dish-3d-comparison.zip', 'w', compression=ZIP_DEFLATED) as archive:
        for rel_path, disk_path in sorted(archived.items()):
            archive.write(disk_path, rel_path)
        archive.writestr('manifest.json', manifest_text)
    (directory / 'manifest.json').write_text(manifest_text, encoding='utf-8')
    return directory / 'manifest.json', directory / 'dish-3d-comparison.zip'

In [ ]:
# Offline fixture checks. Run these separately or before Kaggle GPU execution.
from tempfile import TemporaryDirectory
from unittest.mock import MagicMock, Mock

with BytesIO() as png:
    Image.new('RGBA', (8, 8), (255, 100, 0, 128)).save(png, format='PNG')
    encoded_png = base64.b64encode(png.getvalue()).decode('ascii')
fixture_payload = {
    'choices': [{'message': {'images': [{'image_url': {'url': f'data:image/png;base64,{encoded_png}'}}]}}]
}
with decode_image(fixture_payload) as image:
    assert image.mode == 'RGBA' and image.size == (8, 8)

for invalid in (
    {'choices': []},
    {'error': {'message': 'fixture'}, **fixture_payload},
    {'choices': [{'message': {'images': [{'image_url': {'url': 'https://example.com/dish.png'}}]}}]},
    {'choices': [{'message': {'images': [{'image_url': {'url': f'data:image/jpeg;base64,{encoded_png}'}}]}}]},
    {'choices': [{'message': {'images': [{'image_url': {'url': 'data:image/png;base64,' + base64.b64encode(b'bad').decode()}}]}}]},
):
    try:
        decode_image(invalid)
    except DemoError:
        pass
    else:
        raise AssertionError('Invalid OpenRouter chat image payload was accepted')

# Verify Chat Completions payload, 429 retry, and secret non-leakage
ok_reply = MagicMock()
ok_reply.__enter__.return_value = ok_reply
ok_reply.status_code = 200
ok_reply.iter_content.side_effect = lambda _: [json.dumps(fixture_payload).encode()]

retry_reply = MagicMock()
retry_reply.__enter__.return_value = retry_reply
retry_reply.status_code = 429

post_retry = Mock(side_effect=[retry_reply, ok_reply])
sleeps = []
with request_image('google/gemini-3.1-flash-lite-image', 'fixture-secret', post=post_retry, sleep=sleeps.append) as img:
    assert img.size == (8, 8)
assert post_retry.call_count == 2 and sleeps == [2]
sent_json = post_retry.call_args_list[0].kwargs['json']
assert sent_json['modalities'] == ['image', 'text'] and sent_json['messages'][0]['content'] == DISH_PROMPT

# Preflight DINOv3 unauthorized test
ok, note = preflight_dinov3(None, downloader=Mock(side_effect=RuntimeError('401 GatedRepoError')))
assert ok is False and note == 'missing_or_unauthorized_hf_token'
ok2, note2 = preflight_dinov3('tok', downloader=Mock(return_value='/tmp/config.json'))
assert ok2 is True and note2 == 'ok'

# Verify GLB validator + manifest/ZIP packaging with mixed fallback/OpenRouter provenance
with TemporaryDirectory() as tmp:
    root = Path(tmp)
    img_dir = root / 'sd-turbo'
    img_dir.mkdir(parents=True)
    img_path = img_dir / 'input_image.png'
    Image.new('RGB', (16, 16), (120, 80, 40)).save(img_path)
    inputs = {
        'gemini-3.1-flash-lite-image': {
            'label': 'gemini-3.1-flash-lite-image',
            'model': 'google/gemini-3.1-flash-lite-image',
            'image_source': 'openrouter',
            'image_status': 'failed',
            'image_error': 'OpenRouter HTTP 429.',
        },
        'sd-turbo': {
            'label': 'sd-turbo',
            'model': 'stabilityai/sd-turbo',
            'revision': 'b261bac6fd2cf515557d5d0707481eafa0485ec2',
            'image_source': 'kaggle-open-weights-fallback',
            'image_status': 'success',
            'image_path': img_path,
            'size': [16, 16],
        },
    }
    mesh_json = json.dumps({
        'asset': {'version': '2.0'},
        'meshes': [{'primitives': [{'attributes': {'POSITION': 0, 'COLOR_0': 1}}]}],
        'accessors': [
            {'type': 'VEC3', 'count': 3, 'componentType': 5126},
            {'type': 'VEC3', 'count': 3, 'componentType': 5126},
        ],
    }).encode()
    mesh_json += b' ' * (-len(mesh_json) % 4)
    glb_path = img_dir / 'triposr.glb'
    glb_path.write_bytes(struct.pack('<4sIIII', b'glTF', 2, 20 + len(mesh_json), len(mesh_json), 0x4E4F534A) + mesh_json)
    glb_info = validate_glb(glb_path)
    assert glb_info['has_vertex_color'] is True and glb_info['vertex_count'] == 3

    runs_3d = {
        'sd-turbo/triposr': {
            'input_label': 'sd-turbo',
            'image_source': 'kaggle-open-weights-fallback',
            'backend': 'TripoSR',
            'appearance': 'vertex_color',
            'status': 'success',
            'artifact_paths': [glb_path],
        }
    }
    manifest_path, zip_path = write_results(root, inputs, runs_3d, {'seed': 42}, {'triposr': '107cefdc'})
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert manifest['inputs']['sd-turbo']['image_source'] == 'kaggle-open-weights-fallback'
    assert manifest['inputs']['gemini-3.1-flash-lite-image']['image_status'] == 'failed'
    with ZipFile(zip_path) as archive:
        z_manifest = json.loads(archive.read('manifest.json'))
        for art in z_manifest['runs_3d']['sd-turbo/triposr']['artifacts']:
            assert hashlib.sha256(archive.read(art['path'])).hexdigest() == art['sha256']
    glb_path.write_bytes(b'glTF' + b'\0' * 20)
    try:
        validate_glb(glb_path)
    except DemoError:
        pass
    else:
        raise AssertionError('Invalid GLB was accepted')
print('Offline comparison fixtures passed.')

In [ ]:
import concurrent.futures
import gc
import importlib.metadata
import os
import shutil
import subprocess
import sys
import uuid

ON_KAGGLE = Path('/kaggle').exists()
ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
SCRATCH_ROOT = Path('/tmp/peekaplate_scratch') if ON_KAGGLE else ROOT / '.scratch'
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PYTHON = Path(sys.executable)

HF_CACHE = SCRATCH_ROOT / 'hf_cache'
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TORCH_HOME'] = str(SCRATCH_ROOT / 'torch_cache')
os.environ['U2NET_HOME'] = str(SCRATCH_ROOT / 'u2net')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'

def run(args, **kwargs):
    process = subprocess.Popen(
        [str(v) for v in args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors='replace', bufsize=1, **kwargs
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    ret = process.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed with exit code {ret}: {' '.join(str(a) for a in args)}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a CUDA GPU in Kaggle Settings before continuing.')
GPU_INFO = []
for index in range(torch.cuda.device_count()):
    device = torch.cuda.get_device_properties(index)
    vram_gib = round(device.total_memory / 2**30, 2)
    GPU_INFO.append({'index': index, 'name': device.name, 'vram_gib': vram_gib, 'cc': list(torch.cuda.get_device_capability(index))})
    print(f'GPU {index}: {device.name} ({vram_gib} GiB, cc={GPU_INFO[-1]["cc"]})')
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
GPU_COUNT = torch.cuda.device_count()
MAX_GPU_WORKERS = max(1, min(GPU_COUNT, 2))
print(f'Active CUDA GPUs: {GPU_COUNT}, worker pool size: {MAX_GPU_WORKERS}')

NVCC_AVAILABLE = os.name == 'posix' and shutil.which('nvcc') is not None
if NVCC_AVAILABLE:
    compiler = subprocess.check_output(['nvcc', '--version'], text=True)
    print(compiler.strip())

HF_TOKEN = kaggle_secret('HF_TOKEN') or os.environ.get('HF_TOKEN')
DINOV3_OK, DINOV3_NOTE = preflight_dinov3(HF_TOKEN)
print(f'TRELLIS.2 DINOv3 preflight: ok={DINOV3_OK} ({DINOV3_NOTE})')

# Pin Kaggle preinstalled scientific stack so runtime installs never upgrade numpy/numba mid-kernel
PIN_DISTRIBUTIONS = (
    'numpy', 'scipy', 'torch', 'torchvision', 'numba', 'llvmlite',
    'scikit-image', 'pillow', 'opencv-python-headless', 'transformers',
    'diffusers', 'accelerate',
)
constraints_lines = []
for dist_name in PIN_DISTRIBUTIONS:
    try:
        dist_ver = importlib.metadata.version(dist_name)
        constraints_lines.append(f'{dist_name}=={dist_ver}')
    except importlib.metadata.PackageNotFoundError:
        pass
constraints_path = SCRATCH_ROOT / 'kaggle_constraints.txt'
constraints_path.write_text('\n'.join(constraints_lines) + '\n', encoding='utf-8')

# Install only missing lightweight runtime packages without downgrading/upgrading Kaggle NumPy or Numba
REQUIRED_PACKAGES = [
    ('omegaconf', 'omegaconf>=2.3.0'),
    ('einops', 'einops>=0.7.0'),
    ('trimesh', 'trimesh>=4.6.0'),
    ('rembg', 'rembg[cpu]==2.0.67'),
    ('onnxruntime', 'onnxruntime'),
    ('scipy', 'scipy'),
    ('pymeshlab', 'pymeshlab'),
    ('diffusers', 'diffusers>=0.30.0'),
    ('accelerate', 'accelerate>=0.30.0'),
]
missing_specs = []
for mod_name, spec in REQUIRED_PACKAGES:
    try:
        __import__(mod_name)
    except ImportError:
        missing_specs.append(spec)
if missing_specs:
    print('Installing missing runtime packages with constraints:', missing_specs)
    install_env = os.environ.copy()
    for secret_name in ('OPENROUTER_API_KEY', 'HF_TOKEN'):
        install_env.pop(secret_name, None)
    try:
        run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(constraints_path), *missing_specs], env=install_env)
    except RuntimeError:
        print('Retrying constrained base packages + --no-deps rembg==2.0.67...')
        non_rembg = [s for s in missing_specs if not s.startswith('rembg')]
        extra_deps = [s for s in ('pooch', 'pymatting', 'filetype', 'jsonschema', 'watchdog') if s not in non_rembg]
        run([PYTHON, '-m', 'pip', 'install', '-q', '-c', str(constraints_path), *non_rembg, *extra_deps], env=install_env)
        if any(s.startswith('rembg') for s in missing_specs):
            run([PYTHON, '-m', 'pip', 'install', '-q', '--no-deps', 'rembg==2.0.67'], env=install_env)
else:
    print('All lightweight runtime packages already present.')

REVISIONS = {
    'triposr_repo': '107cefdc244c39106fa830359024f6a2f1c78871',
    'triposr_model': '5b521936b01fbe1890f6f9baed0254ab6351c04a',
    'dino_vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2',
    'hunyuan3d_repo': 'f8db63096c8282cb27354314d896feba5ba6ff8a',
    'hunyuan3d_2mini_model': 'f90a0f7df7d5e6f71109cf333f6a95a0ae3194a6',
    'dinov3': DINOV3_REVISION,
    'trellis_model': TRELLIS_MODEL_REVISION,
    'grounding_dino': GROUNDING_DINO_REVISION,
    'sam_vit_base': SAM_REVISION,
}
DOCKER_LOCK_PACKAGES = (
    'torch', 'torchvision', 'transformers', 'diffusers', 'accelerate',
    'huggingface-hub', 'safetensors', 'omegaconf', 'einops', 'trimesh',
    'pymeshlab', 'rembg', 'onnxruntime', 'scipy', 'scikit-image', 'numpy', 'pillow',
)
RUNTIME_PACKAGE_VERSIONS = {}
for _pkg in DOCKER_LOCK_PACKAGES:
    try:
        RUNTIME_PACKAGE_VERSIONS[_pkg] = importlib.metadata.version(_pkg)
    except importlib.metadata.PackageNotFoundError:
        pass


## 1. Discover user dish photos, preserve the supporting plate, and emit immutable job manifests

Discovers attached Stage 0 dish photos (or falls back only when none exist), isolates food with `birefnet-general-lite`,
preserves the underlying plate/bowl/tray/board via pinned **`grounding-dino-tiny` + `sam-vit-base`** (plus ring-point
prompts and corner leak guards), writes an immutable `job_manifest.json` per dish, and validates cutouts visually.


In [ ]:
import queue
import signal
import threading
from IPython.display import display, Image as DisplayImage, FileLink
import numpy as np


def run_streaming(cmd, env, prefix, timeout=900):
    tail = []
    timed_out = False
    process = subprocess.Popen(
        [str(v) for v in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors='replace',
        bufsize=1,
        env=env,
        start_new_session=True,
    )

    def on_timeout():
        nonlocal timed_out
        timed_out = True
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except OSError:
            pass

    timer = threading.Timer(timeout, on_timeout)
    timer.daemon = True
    timer.start()
    try:
        for line in process.stdout:
            text = line.rstrip('\r\n')
            if text:
                print(f'[{prefix}] {text}', flush=True)
                tail.append(text)
                if len(tail) > 40:
                    del tail[:-40]
        ret = process.wait()
    finally:
        timer.cancel()
        if process.stdout is not None:
            process.stdout.close()
    if timed_out:
        raise subprocess.TimeoutExpired(cmd, timeout, output='\n'.join(tail))
    if ret != 0:
        raise RuntimeError(f'{prefix} exited with code {ret}: ' + '\n'.join(tail[-25:]))
    return '\n'.join(tail)


fallback_worker_code = r'''
import json, sys, time
from pathlib import Path
import torch
from diffusers import StableDiffusionPipeline

cfg = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
t0 = time.perf_counter()
print(f"Loading {cfg['repo']}@{cfg['revision'][:8]} (fp16)...", flush=True)
pipe = StableDiffusionPipeline.from_pretrained(
    cfg['repo'],
    revision=cfg['revision'],
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
    safety_checker=None,
    requires_safety_checker=False,
    token=False,
)
if hasattr(pipe, 'enable_attention_slicing'):
    pipe.enable_attention_slicing()
if hasattr(pipe, 'vae') and hasattr(pipe.vae, 'enable_slicing'):
    pipe.vae.enable_slicing()
pipe = pipe.to('cuda')
print(f"Generating {cfg['size']}x{cfg['size']} image ({cfg['steps']} steps, guidance={cfg['guidance']})...", flush=True)
gen = torch.Generator('cuda').manual_seed(int(cfg['seed']))
kwargs = dict(
    prompt=cfg['prompt'],
    num_inference_steps=int(cfg['steps']),
    guidance_scale=float(cfg['guidance']),
    height=int(cfg['size']),
    width=int(cfg['size']),
    generator=gen,
)
if float(cfg['guidance']) > 0:
    kwargs['negative_prompt'] = 'hands, fork, knife, spoon, text, watermark, cropped rim, multiple plates, collage'
try:
    image = pipe(**kwargs).images[0]
except torch.OutOfMemoryError:
    torch.cuda.empty_cache()
    kwargs['height'] = min(512, int(cfg['size']))
    kwargs['width'] = min(512, int(cfg['size']))
    image = pipe(**kwargs).images[0]
extrema = image.convert('RGB').getextrema()
if all(hi <= 5 for _, hi in extrema):
    raise RuntimeError('Generated image is blank/black.')
out = Path(cfg['output_path'])
out.parent.mkdir(parents=True, exist_ok=True)
image.save(out)
stats = {
    'size': list(image.size),
    'elapsed_seconds': round(time.perf_counter() - t0, 2),
    'gpu_name': torch.cuda.get_device_name(0),
}
Path(cfg['stats_path']).write_text(json.dumps(stats), encoding='utf-8')
print(f"Done in {stats['elapsed_seconds']}s on {stats['gpu_name']}", flush=True)
'''
fallback_worker = SCRATCH_ROOT / 'run_fallback_image.py'
fallback_worker.write_text(fallback_worker_code, encoding='utf-8')

image_results = discover_user_dish_images(RESULTS_DIR)
user_ok = [k for k, v in image_results.items() if v.get('image_status') == 'success']

if user_ok:
    print(f'[stage0] Found {len(user_ok)} user dish photo(s): {user_ok}')
else:
    image_results = collect_openrouter_images(RESULTS_DIR, kaggle_secret('OPENROUTER_API_KEY'))
    for label, entry in image_results.items():
        print('OpenRouter:', entry['model'], entry.get('image_error', entry['image_status']))
    openrouter_ok = [k for k, v in image_results.items() if v['image_status'] == 'success']
    if len(openrouter_ok) < 2:
        needed = 2 - len(openrouter_ok)
        fallback_items = list(FALLBACK_IMAGE_MODELS.items())[:needed]
        print(f'Running {needed} ungated open-weights image fallback model(s) with MAX_GPU_WORKERS={MAX_GPU_WORKERS}...')
        fb_gpu_queue = queue.Queue()
        for gid in range(MAX_GPU_WORKERS):
            fb_gpu_queue.put(gid)

        def run_fallback_image(item):
            label, spec = item
            gpu_idx = fb_gpu_queue.get()
            folder = RESULTS_DIR / label
            folder.mkdir(parents=True, exist_ok=True)
            out_path = folder / 'input_image.png'
            stats_path = SCRATCH_ROOT / f'img_stats_{label}.json'
            cfg_path = SCRATCH_ROOT / f'img_cfg_{label}.json'
            cfg_path.write_text(json.dumps({
                **spec,
                'label': label,
                'prompt': DISH_PROMPT,
                'output_path': str(out_path),
                'stats_path': str(stats_path),
            }), encoding='utf-8')
            env = {k: v for k, v in os.environ.items() if k not in ('OPENROUTER_API_KEY', 'HF_TOKEN')}
            env['CUDA_VISIBLE_DEVICES'] = str(gpu_idx)
            entry = {
                'label': label,
                'model': spec['repo'],
                'revision': spec['revision'],
                'image_source': 'kaggle-open-weights-fallback',
                'directory': folder,
            }
            try:
                run_streaming([PYTHON, fallback_worker, cfg_path], env=env, prefix=f'fallback:{label}', timeout=900)
                stats = json.loads(stats_path.read_text(encoding='utf-8'))
                entry.update(image_status='success', image_path=out_path, **stats)
            except (subprocess.SubprocessError, RuntimeError, OSError, ValueError) as exc:
                print(f'Fallback image {label} failed:', str(exc)[-1200:])
                entry.update(image_status='failed', image_error=type(exc).__name__)
            finally:
                fb_gpu_queue.put(gpu_idx)
            return label, entry

        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_GPU_WORKERS) as pool:
            for label, entry in pool.map(run_fallback_image, fallback_items):
                image_results[label] = entry

ACTIVE_LABELS = [k for k, v in image_results.items() if v.get('image_status') == 'success'][:MAX_USER_DISHES]
if not ACTIVE_LABELS:
    raise RuntimeError('No dish input images succeeded from Stage 0 dataset, OpenRouter, or open-weights fallback.')

cutout_worker_code = r'''
import inspect, json, sys, time
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageOps
from scipy.ndimage import binary_dilation, binary_fill_holes, gaussian_filter, label as nd_label
import rembg

payload = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
jobs = payload['jobs']
stats_out_path = payload.get('stats_out')
dino_repo = payload.get('grounding_dino_repo', 'IDEA-Research/grounding-dino-tiny')
dino_rev = payload.get('grounding_dino_revision')
sam_repo = payload.get('sam_repo', 'facebook/sam-vit-base')
sam_rev = payload.get('sam_revision')
plate_mode = payload.get('plate_mode', 'preserve_plate')

MODEL_ORDER = ('birefnet-general-lite', 'isnet-general-use', 'u2net')
PLATE_PROMPT_PASSES = [
    (
        'a plate. a bowl. a tray. a pan. a platter. a dish. a saucer.',
        [['a plate', 'a bowl', 'a tray', 'a pan', 'a platter', 'a dish', 'a saucer']],
    ),
    (
        'a cutting board. a wooden board. a serving board. a ceramic bowl. a metal tray. a baking pan.',
        [['a cutting board', 'a wooden board', 'a serving board', 'a ceramic bowl', 'a metal tray', 'a baking pan']],
    ),
]
rembg_sessions = {}
plate_models = {}


def get_rembg_session(model_name):
    if model_name not in rembg_sessions:
        print(f'Initializing rembg session: {model_name} (CPU)...', flush=True)
        rembg_sessions[model_name] = rembg.new_session(model_name, providers=['CPUExecutionProvider'])
    return rembg_sessions[model_name]


def get_plate_models():
    if not plate_models:
        from transformers import (
            AutoModelForZeroShotObjectDetection,
            AutoProcessor,
            SamModel,
            SamProcessor,
        )
        device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        print(f'Loading GroundingDINO@{str(dino_rev)[:8]} + SAM@{str(sam_rev)[:8]} on {device}...', flush=True)
        dino_proc = AutoProcessor.from_pretrained(dino_repo, revision=dino_rev, token=False)
        dino_mod = AutoModelForZeroShotObjectDetection.from_pretrained(
            dino_repo, revision=dino_rev, token=False
        ).eval().to(device)
        sam_proc = SamProcessor.from_pretrained(sam_repo, revision=sam_rev, token=False)
        sam_mod = SamModel.from_pretrained(
            sam_repo, revision=sam_rev, token=False
        ).eval().to(device)
        plate_models.update(
            device=device,
            dino_proc=dino_proc,
            dino_mod=dino_mod,
            sam_proc=sam_proc,
            sam_mod=sam_mod,
        )
    return plate_models


def extract_food_alpha(rgb, label_name):
    last_err = None
    for model_name in MODEL_ORDER:
        try:
            sess = get_rembg_session(model_name)
            rgba = rembg.remove(rgb, session=sess)
            arr = np.array(rgba.convert('RGBA'), dtype=np.uint8)
            alpha = arr[:, :, 3]
            raw_mask = alpha > 16
            filled = binary_fill_holes(raw_mask)
            labeled, num_features = nd_label(filled)
            if num_features > 1:
                counts = np.bincount(labeled.ravel())[1:]
                max_area = int(counts.max())
                keep = np.zeros(num_features + 1, dtype=bool)
                for cid, area in enumerate(counts, start=1):
                    if area >= max(64, int(0.005 * max_area)):
                        keep[cid] = True
                clean_mask = keep[labeled]
            else:
                clean_mask = filled
            fg_check = clean_mask & (alpha >= 128)
            ys, xs = np.where(fg_check)
            if len(ys) == 0:
                ys, xs = np.where(clean_mask)
            if len(ys) == 0 or (xs.max() - xs.min()) < 8 or (ys.max() - ys.min()) < 8:
                raise RuntimeError(f'{model_name} produced empty food foreground')
            food_alpha = np.where(clean_mask, np.where(clean_mask & (alpha <= 16), 255, alpha), 0).astype(np.uint8)
            return food_alpha, clean_mask, model_name, int(num_features)
        except Exception as exc:
            last_err = exc
            print(f'rembg {model_name} fallback on {label_name}: {type(exc).__name__}: {exc}', flush=True)
    raise RuntimeError(f'All rembg models failed for {label_name}: {last_err}')


def _corner_block_pair(h, w, patch=8):
    p_h = min(patch, max(1, h // 8))
    p_w = min(patch, max(1, w // 8))
    ins = min(max(8, int(round(0.035 * min(h, w)))), max(0, min(h, w) // 4))
    p_in = min(max(patch, int(round(0.04 * min(h, w)))), max(1, min(h, w) // 4))
    pairs = (
        ((slice(0, p_h), slice(0, p_w)), (slice(ins, ins + p_in), slice(ins, ins + p_in))),
        ((slice(0, p_h), slice(w - p_w, w)), (slice(ins, ins + p_in), slice(w - ins - p_in, w - ins))),
        ((slice(h - p_h, h), slice(0, p_w)), (slice(h - ins - p_in, h - ins), slice(ins, ins + p_in))),
        ((slice(h - p_h, h), slice(w - p_w, w)), (slice(h - ins - p_in, h - ins), slice(w - ins - p_in, w - ins))),
    )
    return pairs


def count_non_food_corners(mask, food_mask, patch=8):
    h, w = mask.shape
    non_food = mask & (~food_mask)
    hits = 0
    for (oy, ox), (iy, ix) in _corner_block_pair(h, w, patch):
        if float(non_food[oy, ox].mean()) > 0.50 or float(non_food[iy, ix].mean()) > 0.55:
            hits += 1
    return hits


def count_top_non_food_corners(mask, food_mask, patch=8):
    h, w = mask.shape
    non_food = mask & (~food_mask)
    hits = 0
    for (oy, ox), (iy, ix) in _corner_block_pair(h, w, patch)[:2]:
        if float(non_food[oy, ox].mean()) > 0.50 or float(non_food[iy, ix].mean()) > 0.55:
            hits += 1
    return hits


def _box_iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0.0:
        return 0.0
    area_a = max(0.0, (a[2] - a[0]) * (a[3] - a[1]))
    area_b = max(0.0, (b[2] - b[0]) * (b[3] - b[1]))
    return inter / max(area_a + area_b - inter, 1e-6)


def detect_plate_mask(rgb, food_mask, label_name='dish'):
    w, h = rgb.size
    models = get_plate_models()
    device = models['device']
    dino_proc = models['dino_proc']
    dino_mod = models['dino_mod']
    sam_proc = models['sam_proc']
    sam_mod = models['sam_mod']

    fys, fxs = np.where(food_mask)
    fx1, fy1 = int(fxs.min()), int(fys.min())
    fx2, fy2 = int(fxs.max()), int(fys.max())
    fw = max(fx2 - fx1 + 1, 1)
    fh = max(fy2 - fy1 + 1, 1)
    fcx = float(fxs.mean())
    fcy = float(fys.mean())
    food_sum = max(int(food_mask.sum()), 1)
    food_ring = binary_dilation(food_mask, iterations=max(4, int(round(0.015 * min(h, w))))) & (~food_mask)

    sig_params = inspect.signature(dino_proc.post_process_grounded_object_detection).parameters
    scored_boxes = []
    for prompt_text, prompt_labels in PLATE_PROMPT_PASSES:
        dino_inputs = dino_proc(images=rgb, text=prompt_text, return_tensors='pt').to(device)
        with torch.inference_mode():
            dino_out = dino_mod(**dino_inputs)
        pp_kwargs = {'target_sizes': [(h, w)]}
        if 'threshold' in sig_params:
            pp_kwargs['threshold'] = 0.15
        elif 'box_threshold' in sig_params:
            pp_kwargs['box_threshold'] = 0.15
        if 'text_threshold' in sig_params:
            pp_kwargs['text_threshold'] = 0.13
        if 'text_labels' in sig_params:
            pp_kwargs['text_labels'] = prompt_labels
        det = dino_proc.post_process_grounded_object_detection(
            dino_out, input_ids=dino_inputs['input_ids'], **pp_kwargs
        )[0]
        raw_boxes = det['boxes'].detach().cpu().tolist()
        raw_scores = det['scores'].detach().cpu().tolist()
        for box, sc in zip(raw_boxes, raw_scores):
            x1, y1, x2, y2 = [float(v) for v in box]
            bx1, by1 = max(0, int(np.floor(x1))), max(0, int(np.floor(y1)))
            bx2, by2 = min(w, int(np.ceil(x2))), min(h, int(np.ceil(y2)))
            if bx2 - bx1 < 16 or by2 - by1 < 16:
                continue
            overlap = float(food_mask[by1:by2, bx1:bx2].sum()) / food_sum
            contains_center = (x1 <= fcx <= x2) and (y1 <= fcy <= y2)
            if overlap >= 0.25 or contains_center:
                box_area_frac = ((bx2 - bx1) * (by2 - by1)) / float(h * w)
                b_sc = float(sc) + 0.30 * overlap + (0.12 if box_area_frac > food_mask.mean() else 0.0) - 0.15 * max(0.0, box_area_frac - 0.88)
                scored_boxes.append((b_sc, [x1, y1, x2, y2]))

    scored_boxes.sort(key=lambda pair: pair[0], reverse=True)
    selected_boxes = []
    for _, cand_box in scored_boxes:
        if all(_box_iou(cand_box, kept) <= 0.90 for kept in selected_boxes):
            selected_boxes.append(cand_box)
        if len(selected_boxes) >= 6:
            break

    candidates = []

    def evaluate_sam_masks(masks_batch, iou_batch, source_tag):
        for k_idx in range(masks_batch.shape[0]):
            for m_idx in range(masks_batch.shape[1]):
                m = masks_batch[k_idx, m_idx] > 0.0
                if not m.any():
                    continue
                if float(m.mean()) > 0.92:
                    continue
                if count_non_food_corners(m, food_mask) >= 2 or count_top_non_food_corners(m, food_mask) >= 2:
                    continue
                if float((m[0, :] & (~food_mask[0, :])).mean()) > 0.50:
                    continue
                if float((m[:, 0] & (~food_mask[:, 0])).mean()) > 0.72 and float((m[:, -1] & (~food_mask[:, -1])).mean()) > 0.72:
                    continue
                iou_sc = float(iou_batch[k_idx, m_idx])
                u_raw = binary_fill_holes(m | food_mask)
                lbl_u, n_u = nd_label(u_raw)
                if n_u > 1:
                    keep_u = np.zeros(n_u + 1, dtype=bool)
                    for cid in range(1, n_u + 1):
                        comp = lbl_u == cid
                        if (comp & food_mask).any():
                            keep_u[cid] = True
                    u = keep_u[lbl_u]
                else:
                    u = u_raw
                plate_only = u & (~food_mask)
                plate_gain = float(plate_only.sum()) / float(h * w)
                union_area_frac = float(u.mean())
                u_corners = count_non_food_corners(u, food_mask)
                u_top_corners = count_top_non_food_corners(u, food_mask)
                u_top_edge = float((u[0, :] & (~food_mask[0, :])).mean())
                ring_contact = float((plate_only & food_ring).sum()) / max(int(food_ring.sum()), 1)
                raw_touch = float((binary_dilation(m, iterations=3) & food_mask).sum()) / food_sum
                u_ys, u_xs = np.where(u)
                if len(u_ys) == 0:
                    continue
                bbox_fill = float(u.sum()) / max(float((u_xs.max() - u_xs.min() + 1) * (u_ys.max() - u_ys.min() + 1)), 1.0)
                u_left_edge = float((u[:, 0] & (~food_mask[:, 0])).mean())
                u_right_edge = float((u[:, -1] & (~food_mask[:, -1])).mean())
                if (
                    u_corners >= 2
                    or u_top_corners >= 2
                    or u_top_edge > 0.55
                    or (u_left_edge > 0.72 and u_right_edge > 0.72)
                    or union_area_frac > 0.92
                    or plate_gain < 0.018
                    or ring_contact < 0.25
                    or raw_touch < 0.10
                    or bbox_fill < 0.48
                ):
                    continue
                score = (
                    0.70 * iou_sc
                    + 0.85 * ring_contact
                    + 0.70 * bbox_fill
                    + min(plate_gain, 0.38) * 2.2
                    + (0.10 if source_tag == 'grounding_dino_sam' else 0.0)
                    - 0.55 * u_corners
                    - 0.30 * max(0.0, union_area_frac - 0.85)
                )
                meta = {
                    'iou': round(iou_sc, 3),
                    'gain': round(plate_gain, 3),
                    'ring': round(ring_contact, 3),
                    'fill': round(bbox_fill, 3),
                    'area': round(union_area_frac, 3),
                }
                candidates.append((score, source_tag, u_corners, u, meta))

    def run_sam_point_prompt(pos_pts, neg_pts, source_tag):
        if not pos_pts:
            return
        pts = pos_pts + neg_pts
        lbls = [1] * len(pos_pts) + [0] * len(neg_pts)
        sam_pt_inputs = sam_proc(images=rgb, input_points=[pts], input_labels=[lbls], return_tensors='pt').to(device)
        with torch.inference_mode():
            sam_pt_out = sam_mod(**sam_pt_inputs, multimask_output=True)
        pt_post = sam_proc.image_processor.post_process_masks(
            sam_pt_out.pred_masks.cpu(),
            sam_pt_inputs['original_sizes'].cpu(),
            sam_pt_inputs['reshaped_input_sizes'].cpu(),
        )[0].numpy()
        pt_ious = sam_pt_out.iou_scores[0].detach().cpu().numpy()
        evaluate_sam_masks(pt_post, pt_ious, source_tag)

    # 1. GroundingDINO boxes -> SAM
    if selected_boxes:
        sam_inputs = sam_proc(images=rgb, input_boxes=[selected_boxes], return_tensors='pt').to(device)
        with torch.inference_mode():
            sam_out = sam_mod(**sam_inputs, multimask_output=True)
        post = sam_proc.image_processor.post_process_masks(
            sam_out.pred_masks.cpu(),
            sam_inputs['original_sizes'].cpu(),
            sam_inputs['reshaped_input_sizes'].cpu(),
        )[0].numpy()
        ious = sam_out.iou_scores[0].detach().cpu().numpy()
        evaluate_sam_masks(post, ious, 'grounding_dino_sam')

    pad_c = 4
    ins_c = min(max(12, int(round(0.035 * min(h, w)))), max(1, min(h, w) // 4))
    corner_negs = []
    for cx, cy in (
        (pad_c, pad_c), (w - 1 - pad_c, pad_c),
        (pad_c, h - 1 - pad_c), (w - 1 - pad_c, h - 1 - pad_c),
        (ins_c, ins_c), (w - 1 - ins_c, ins_c),
    ):
        ix, iy = int(np.clip(cx, 0, w - 1)), int(np.clip(cy, 0, h - 1))
        if not food_mask[iy, ix]:
            corner_negs.append([float(ix), float(iy)])

    # 2. Directional lower/lateral support-ring SAM prompts (avoids protruding top garnish)
    r_in = max(4, int(round(0.018 * min(h, w))))
    r_out = max(14, int(round(0.075 * min(h, w))))
    ring_band = binary_dilation(food_mask, iterations=r_out) & (~binary_dilation(food_mask, iterations=r_in))
    y_floor = min(h - 9, fy1 + int(round(0.25 * fh)))
    valid_ring = ring_band.copy()
    valid_ring[:max(8, y_floor), :] = False
    valid_ring[max(8, h - 8):, :] = False
    valid_ring[:, :8] = False
    valid_ring[:, max(8, w - 8):] = False
    ry, rx = np.where(valid_ring)
    if len(ry) >= 4:
        angles = np.arctan2(ry.astype(np.float64) - fcy, rx.astype(np.float64) - fcx)
        order = np.argsort(angles)
        picks = order[np.linspace(0, len(order) - 1, min(6, len(order)), dtype=int)]
        ring_pos = [[float(rx[idx]), float(ry[idx])] for idx in picks]
        run_sam_point_prompt(ring_pos, corner_negs, 'sam_ring_points')
        interior_pos = [[float(int(np.clip(fcx, 2, w - 3))), float(int(np.clip(fcy, 2, h - 3)))]]
        low_y = int(np.clip(fy1 + int(round(0.65 * fh)), 2, h - 3))
        low_xs = np.where(food_mask[low_y])[0]
        if len(low_xs) > 0:
            interior_pos.append([float(int(low_xs.mean())), float(low_y)])
        else:
            interior_pos.append([float(int(np.clip(fcx, 2, w - 3))), float(low_y)])
        run_sam_point_prompt(ring_pos + interior_pos, corner_negs, 'sam_ring_points')

    # 3. Direct vessel-surface single-click & pair-click probes around lower/lateral perimeter
    vessel_pts = []
    step_lat = max(10, int(round(0.055 * min(h, w))))
    pt_bx = int(np.clip(round(fcx), 8, w - 9))
    pt_by = int(np.clip(fy2 + max(8, int(round(0.045 * h))), 8, h - 9))
    if not food_mask[pt_by, pt_bx]:
        vessel_pts.append([float(pt_bx), float(pt_by)])

    yr = int(np.clip(fy1 + int(round(0.55 * fh)), 8, h - 9))
    rxs = np.where(food_mask[yr])[0]
    if len(rxs) > 0:
        px = min(w - 10, int(rxs.max()) + step_lat)
        while px < w - 8 and food_mask[yr, px]:
            px += 8
        if 8 <= px <= w - 9 and not food_mask[yr, px]:
            vessel_pts.append([float(px), float(yr)])

    yl = int(np.clip(fy1 + int(round(0.65 * fh)), 8, h - 9))
    lxs = np.where(food_mask[yl])[0]
    if len(lxs) > 0:
        px = max(9, int(lxs.min()) - step_lat)
        while px > 8 and food_mask[yl, px]:
            px -= 8
        if 8 <= px <= w - 9 and not food_mask[yl, px]:
            vessel_pts.append([float(px), float(yl)])

    for vp in vessel_pts:
        run_sam_point_prompt([vp], corner_negs, 'sam_vessel_points')
    if len(vessel_pts) >= 2:
        run_sam_point_prompt(vessel_pts[:2], corner_negs, 'sam_vessel_points')

    if not candidates:
        print(f'Plate candidates [{label_name}]: none passed guards (boxes={len(selected_boxes)})', flush=True)
        return None, 'food_only_fallback', 0, len(selected_boxes)

    candidates.sort(key=lambda item: item[0], reverse=True)
    top3 = [f'{src}:{round(sc, 3)}({meta})' for sc, src, _, _, meta in candidates[:3]]
    print(f'Plate candidates [{label_name}] boxes={len(selected_boxes)} top3={top3}', flush=True)
    _, method, corners_hit, best_mask, _ = candidates[0]
    return best_mask, method, int(corners_hit), len(selected_boxes)


all_stats = {}
for job in jobs:
    t0 = time.perf_counter()
    with Image.open(job['src']) as im:
        rgb = ImageOps.exif_transpose(im).convert('RGB')
    w, h = rgb.size
    rgb_arr = np.array(rgb, dtype=np.uint8)

    reviewed_path = job.get('reviewed_mask')
    if reviewed_path and Path(reviewed_path).is_file():
        with Image.open(reviewed_path) as rm:
            mask_l = ImageOps.exif_transpose(rm).convert('L').resize((w, h), Image.Resampling.BILINEAR)
        new_alpha = np.array(mask_l, dtype=np.uint8)
        clean_mask = new_alpha >= 128
        food_coverage_pct = round(float(clean_mask.mean()) * 100.0, 2)
        plate_preserved = True
        plate_method = 'reviewed_mask'
        plate_added_pct = 0.0
        corners_hit = count_non_food_corners(clean_mask, clean_mask)
        det_boxes = 0
        used_model = 'reviewed-full-plate-mask'
        comps_before = 1
    else:
        food_alpha, food_mask, used_model, comps_before = extract_food_alpha(rgb, job['label'])
        food_coverage_pct = round(float((food_alpha >= 128).mean()) * 100.0, 2)
        plate_mask = None
        plate_method = 'food_only'
        corners_hit = 0
        det_boxes = 0
        if plate_mode == 'preserve_plate':
            try:
                plate_mask, plate_method, corners_hit, det_boxes = detect_plate_mask(rgb, food_mask, job['label'])
            except Exception as exc:
                print(f'Plate segmentation warning on {job["label"]}: {type(exc).__name__}: {exc}', flush=True)
                plate_mask, plate_method = None, 'food_only_fallback'

        if plate_mask is not None:
            union_raw = binary_fill_holes(food_mask | plate_mask)
            labeled_u, n_u = nd_label(union_raw)
            if n_u > 1:
                keep_u = np.zeros(n_u + 1, dtype=bool)
                for cid in range(1, n_u + 1):
                    comp = labeled_u == cid
                    if (comp & food_mask).any():
                        keep_u[cid] = True
                clean_mask = keep_u[labeled_u]
            else:
                clean_mask = union_raw
            plate_only = clean_mask & (~food_mask)
            plate_edge = gaussian_filter(clean_mask.astype(np.float32), sigma=1.0)
            plate_alpha = np.where(clean_mask, np.clip(np.round(plate_edge * 255.0), 210, 255), 0).astype(np.uint8)
            new_alpha = np.where(food_mask, food_alpha, np.where(plate_only, plate_alpha, 0)).astype(np.uint8)
            plate_added_pct = round(float((plate_only & (new_alpha >= 128)).mean()) * 100.0, 2)
            plate_preserved = plate_added_pct > 0.0
        else:
            clean_mask = food_mask
            new_alpha = food_alpha
            plate_added_pct = 0.0
            plate_preserved = False

    fg_bin = new_alpha >= 128
    ys, xs = np.where(fg_bin)
    if len(ys) == 0:
        ys, xs = np.where(new_alpha > 16)
    if len(ys) == 0 or (xs.max() - xs.min()) < 8 or (ys.max() - ys.min()) < 8:
        raise RuntimeError(f"Cutout produced empty foreground for {job['label']}")

    rgba_arr = np.dstack([rgb_arr, new_alpha]).astype(np.uint8)
    rgba_arr[new_alpha == 0, :3] = 0
    clean_rgba = Image.fromarray(rgba_arr, mode='RGBA')
    clean_rgba.save(job['dst'])

    _, comps_after = nd_label(new_alpha > 16)
    edge_sides = []
    if bool((new_alpha[:, 0] >= 128).any()):
        edge_sides.append('left')
    if bool((new_alpha[:, -1] >= 128).any()):
        edge_sides.append('right')
    if bool((new_alpha[0, :] >= 128).any()):
        edge_sides.append('top')
    if bool((new_alpha[-1, :] >= 128).any()):
        edge_sides.append('bottom')
    bw = int(xs.max() - xs.min() + 1)
    bh = int(ys.max() - ys.min() + 1)

    job_stats = {
        'cutout_model': used_model,
        'plate_mode': plate_mode,
        'plate_preserved': bool(plate_preserved),
        'plate_method': plate_method,
        'food_coverage_pct': food_coverage_pct,
        'plate_added_pct': plate_added_pct,
        'fg_coverage_pct': round(float(fg_bin.mean()) * 100.0, 2),
        'bbox_aspect': round(bw / max(bh, 1), 3),
        'bbox': [int(xs.min()), int(ys.min()), int(xs.max() + 1), int(ys.max() + 1)],
        'edge_touch_sides': edge_sides,
        'corners_hit': int(corners_hit),
        'detected_plate_boxes': int(det_boxes),
        'components_before': int(comps_before),
        'components_after': int(comps_after),
        'elapsed_seconds': round(time.perf_counter() - t0, 2),
    }
    all_stats[job['label']] = job_stats
    print(
        f"Cutout [{job['label']}] model={used_model} plate={plate_method}({plate_preserved}) "
        f"food={food_coverage_pct}% +plate={plate_added_pct}% -> fg={job_stats['fg_coverage_pct']}% "
        f"corners={corners_hit} edges={edge_sides} ({job_stats['elapsed_seconds']}s)",
        flush=True,
    )

plate_models.clear()
rembg_sessions.clear()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if stats_out_path:
    Path(stats_out_path).write_text(json.dumps(all_stats, indent=2), encoding='utf-8')
'''
cutout_worker = SCRATCH_ROOT / 'run_rembg_cutout.py'
cutout_worker.write_text(cutout_worker_code, encoding='utf-8')
cutout_jobs = [
    {
        'label': label,
        'src': str(image_results[label]['image_path']),
        'dst': str(Path(image_results[label]['image_path']).parent / 'cutout_rgba.png'),
        'reviewed_mask': str(image_results[label]['reviewed_mask_path']) if image_results[label].get('reviewed_mask_path') else None,
    }
    for label in ACTIVE_LABELS
]
cutout_stats_path = SCRATCH_ROOT / 'cutout_stats.json'
cutout_cfg = SCRATCH_ROOT / 'cutout_jobs.json'
cutout_cfg.write_text(json.dumps({
    'jobs': cutout_jobs,
    'stats_out': str(cutout_stats_path),
    'grounding_dino_repo': GROUNDING_DINO_REPO,
    'grounding_dino_revision': GROUNDING_DINO_REVISION,
    'sam_repo': SAM_REPO,
    'sam_revision': SAM_REVISION,
    'plate_mode': PLATE_PRESERVATION_MODE,
}), encoding='utf-8')
clean_env = {k: v for k, v in os.environ.items() if k not in ('OPENROUTER_API_KEY', 'HF_TOKEN')}
clean_env['CUDA_VISIBLE_DEVICES'] = '0'
run_streaming([PYTHON, cutout_worker, cutout_cfg], env=clean_env, prefix='rembg', timeout=600)
cutout_stats = json.loads(cutout_stats_path.read_text(encoding='utf-8'))

for job in cutout_jobs:
    label = job['label']
    entry = image_results[label]
    entry['cutout_path'] = Path(job['dst'])
    entry['cutout_metrics'] = cutout_stats[label]
    REVISIONS[f'input_{label}'] = entry.get('revision', entry['image_source'])
    REVISIONS[f'cutout_{label}'] = f"{cutout_stats[label]['cutout_model']}+{cutout_stats[label]['plate_method']}"
    jm = build_job_manifest(label, entry, REVISIONS)
    jm_path = Path(entry['image_path']).parent / 'job_manifest.json'
    jm_path.write_text(json.dumps(jm, indent=2) + '\n', encoding='utf-8')
    entry['job_manifest_path'] = jm_path
    print(f"Active dish [{label}] job_id={jm['job_id']} cutout={entry['cutout_metrics']}")
    display(DisplayImage(filename=job['src'], width=280))
    display(DisplayImage(filename=job['dst'], width=280))


## 2. Parallel `TripoSR` reconstruction (Raw + Mobile-Optimized `<=35k` GLB, ungated)

Runs `VAST-AI-Research/TripoSR` at `resolution=256` (`192` OOM fallback) on the plate-preserving cutout,
rotates vertices (`[:, [1, 2, 0]]`) into glTF `+Y` up / `+Z` front, prunes tiny floating fragments, and exports
both **`triposr.glb`** and quadric-decimated **`triposr_mobile.glb`** (`<=35k` faces with vertex colors preserved).


In [ ]:
import queue

TRIPOSR_DIR = SCRATCH_ROOT / 'TripoSR-src'
if not TRIPOSR_DIR.exists():
    run(['git', 'init', '-q', str(TRIPOSR_DIR)])
    run(['git', '-C', str(TRIPOSR_DIR), 'fetch', '-q', '--depth', '1',
         'https://github.com/VAST-AI-Research/TripoSR.git', REVISIONS['triposr_repo']])
    run(['git', '-C', str(TRIPOSR_DIR), 'checkout', '-q', '--detach', 'FETCH_HEAD'])

from huggingface_hub import hf_hub_download
dino_vitb_cfg = hf_hub_download('facebook/dino-vitb16', 'config.json', revision=REVISIONS['dino_vitb16'], token=False)
for fname in ('config.yaml', 'model.ckpt'):
    triposr_ckpt_dir = Path(hf_hub_download('stabilityai/TripoSR', fname, revision=REVISIONS['triposr_model'], token=False)).parent

triposr_worker_code = r'''
import json, os, sys, time, types
from pathlib import Path
import numpy as np
import pymeshlab
import torch
import trimesh
from PIL import Image
from skimage.measure import marching_cubes

cfg = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
sys.path.insert(0, cfg['source_dir'])

def marching_cubes_cpu(level, threshold):
    volume = level.detach().float().cpu().numpy()
    if not np.isfinite(volume).all() or not volume.min() < threshold < volume.max():
        raise ValueError('TripoSR did not predict an extractable density surface.')
    vertices, faces, _, _ = marching_cubes(
        volume, level=threshold, gradient_direction='ascent', allow_degenerate=False,
    )
    return (
        torch.from_numpy(vertices[:, ::-1].copy()),
        torch.from_numpy(faces.astype(np.int64)),
    )

mcubes = types.ModuleType('torchmcubes')
mcubes.marching_cubes = marching_cubes_cpu
sys.modules['torchmcubes'] = mcubes

from tsr.models.tokenizers import image as image_tokenizer
from tsr.system import TSR
from tsr.utils import resize_foreground

image_tokenizer.hf_hub_download = lambda *args, **kwargs: cfg['dino_cfg']
with Image.open(cfg['cutout_path']) as rgba_im:
    rgba = resize_foreground(rgba_im.convert('RGBA'), 0.85)
gray = Image.new('RGBA', rgba.size, (128, 128, 128, 255))
rgb = Image.alpha_composite(gray, rgba).convert('RGB')

device = torch.device('cuda:0')
t0 = time.perf_counter()
def remap_triposr_vit_state_dict(ckpt, target_keys):
    prefix = 'image_tokenizer.model.'
    if (f'{prefix}layers.0.attention.q_proj.weight' not in target_keys
            or f'{prefix}encoder.layer.0.attention.attention.query.weight' not in ckpt):
        return ckpt
    remapped = {}
    for key, val in ckpt.items():
        new_key = key
        if key.startswith(f'{prefix}encoder.layer.'):
            new_key = key.replace(f'{prefix}encoder.layer.', f'{prefix}layers.', 1)
            for old_part, new_part in (
                ('.attention.attention.query.', '.attention.q_proj.'),
                ('.attention.attention.key.', '.attention.k_proj.'),
                ('.attention.attention.value.', '.attention.v_proj.'),
                ('.attention.output.dense.', '.attention.o_proj.'),
                ('.intermediate.dense.', '.mlp.fc1.'),
                ('.output.dense.', '.mlp.fc2.'),
            ):
                if old_part in new_key:
                    new_key = new_key.replace(old_part, new_part, 1)
                    break
        remapped[new_key] = val
    return remapped

from omegaconf import OmegaConf
print('Loading TripoSR model onto GPU...', flush=True)
tsr_cfg = OmegaConf.load(str(Path(cfg['ckpt_dir']) / 'config.yaml'))
OmegaConf.resolve(tsr_cfg)
model = TSR(tsr_cfg)
ckpt = torch.load(str(Path(cfg['ckpt_dir']) / 'model.ckpt'), map_location='cpu')
ckpt = remap_triposr_vit_state_dict(ckpt, model.state_dict().keys())
model.load_state_dict(ckpt, strict=True)
model = model.eval().to(device)
model.renderer.set_chunk_size(8192)

primary_res = int(cfg['resolution'])
fallback_res = 192 if primary_res > 192 else 160
res_candidates = (primary_res,) if primary_res == fallback_res else (primary_res, fallback_res)
used_res = None
mesh = None
for res in res_candidates:
    try:
        with torch.inference_mode():
            scene = model([rgb], device=str(device))
            mesh, = model.extract_mesh(scene, has_vertex_color=True, resolution=res)
        used_res = res
        break
    except torch.OutOfMemoryError:
        torch.cuda.empty_cache()
        if res == res_candidates[-1]:
            raise

if not len(mesh.faces) or not np.isfinite(mesh.vertices).all() or mesh.visual.kind != 'vertex':
    raise ValueError('TripoSR returned invalid mesh geometry or missing vertex colors.')

# Rotate TripoSR canonical (X_front, Y_right, Z_up) -> glTF (+Y up, +Z front, det=+1 preserves outward winding)
rot_verts = np.ascontiguousarray(mesh.vertices[:, [1, 2, 0]], dtype=np.float64)
vc_arr = np.ascontiguousarray(mesh.visual.vertex_colors)
mesh = trimesh.Trimesh(vertices=rot_verts, faces=mesh.faces.copy(), vertex_colors=vc_arr, process=False)

components_before = 1
components_after = 1
try:
    parts = mesh.split(only_watertight=False)
    components_before = len(parts)
    if components_before > 1:
        counts = [len(part.faces) for part in parts]
        max_faces = max(counts)
        keep = [part for part, fc in zip(parts, counts) if fc >= max(16, int(0.01 * max_faces))]
        if keep:
            pruned = keep[0] if len(keep) == 1 else trimesh.util.concatenate(keep)
            if len(pruned.faces) > 0 and np.isfinite(pruned.vertices).all() and pruned.visual.kind == 'vertex':
                mesh = pruned
                components_after = len(keep)
            else:
                components_after = components_before
        else:
            components_after = components_before
except Exception as exc:
    print(f'TripoSR component pruning skipped ({type(exc).__name__}): {exc}', flush=True)

if not len(mesh.faces) or not np.isfinite(mesh.vertices).all() or mesh.visual.kind != 'vertex':
    raise ValueError('TripoSR post-processing returned invalid mesh geometry or missing vertex colors.')

out_glb = Path(cfg['output_glb'])
out_glb.parent.mkdir(parents=True, exist_ok=True)
out_glb.write_bytes(mesh.export(file_type='glb'))

mobile_stats = None
if cfg.get('mobile_glb'):
    mobile_glb = Path(cfg['mobile_glb'])
    target_faces = min(int(cfg.get('mobile_max_faces', 34000)), int(len(mesh.faces)))
    ms = pymeshlab.MeshSet()
    vc = np.asarray(mesh.visual.vertex_colors, dtype=np.float64) / 255.0
    if vc.shape[1] == 3:
        vc = np.concatenate([vc, np.ones((len(vc), 1))], axis=1)
    pm = pymeshlab.Mesh(
        vertex_matrix=np.asarray(mesh.vertices, dtype=np.float64),
        face_matrix=np.asarray(mesh.faces, dtype=np.int32),
        v_color_matrix=vc,
    )
    ms.add_mesh(pm)
    ms.apply_filter('meshing_decimation_quadric_edge_collapse', targetfacenum=target_faces, preserveboundary=True)
    cm = ms.current_mesh()
    m_verts = np.ascontiguousarray(cm.vertex_matrix(), dtype=np.float64)
    m_faces = np.ascontiguousarray(cm.face_matrix(), dtype=np.int64)
    if not cm.has_vertex_color():
        raise ValueError('TripoSR mobile decimation dropped vertex colors.')
    m_vc = (np.clip(cm.vertex_color_matrix()[:, :4], 0.0, 1.0) * 255.0).round().astype(np.uint8)
    mobile_mesh = trimesh.Trimesh(vertices=m_verts, faces=m_faces, vertex_colors=m_vc, process=False)
    if not len(mobile_mesh.faces) or not np.isfinite(mobile_mesh.vertices).all() or mobile_mesh.visual.kind != 'vertex':
        raise ValueError('TripoSR mobile decimation produced invalid mesh geometry.')
    mobile_glb.parent.mkdir(parents=True, exist_ok=True)
    mobile_glb.write_bytes(mobile_mesh.export(file_type='glb'))
    mobile_stats = {'vertices': int(len(mobile_mesh.vertices)), 'faces': int(len(mobile_mesh.faces))}

bounds = mesh.bounds.tolist() if mesh.bounds is not None else None
stats = {
    'vertices': int(len(mesh.vertices)),
    'faces': int(len(mesh.faces)),
    'bounds': bounds,
    'resolution': used_res,
    'components_before': components_before,
    'components_after': components_after,
    'mobile': mobile_stats,
    'elapsed_seconds': round(time.perf_counter() - t0, 2),
    'gpu_name': torch.cuda.get_device_name(0),
    'peak_vram_gib': round(torch.cuda.max_memory_allocated(0) / 2**30, 3),
}
Path(cfg['stats_path']).write_text(json.dumps(stats), encoding='utf-8')
print(
    f"TripoSR exported v={stats['vertices']} f={stats['faces']} "
    f"(mobile f={mobile_stats['faces'] if mobile_stats else '-'}) "
    f"comps={components_before}->{components_after} res={used_res} in {stats['elapsed_seconds']}s",
    flush=True,
)
'''
triposr_worker = SCRATCH_ROOT / 'run_triposr_worker.py'
triposr_worker.write_text(triposr_worker_code, encoding='utf-8')

RUNS_3D = {}
triposr_gpu_queue = queue.Queue()
for gid in range(MAX_GPU_WORKERS):
    triposr_gpu_queue.put(gid)

def run_triposr_for(label):
    gpu_idx = triposr_gpu_queue.get()
    in_entry = image_results[label]
    out_glb = RESULTS_DIR / label / 'triposr.glb'
    mobile_glb = RESULTS_DIR / label / 'triposr_mobile.glb'
    stats_path = SCRATCH_ROOT / f'triposr_stats_{label}.json'
    cfg_path = SCRATCH_ROOT / f'triposr_cfg_{label}.json'
    cfg_path.write_text(json.dumps({
        'source_dir': str(TRIPOSR_DIR),
        'ckpt_dir': str(triposr_ckpt_dir),
        'dino_cfg': str(dino_vitb_cfg),
        'cutout_path': str(in_entry['cutout_path']),
        'output_glb': str(out_glb),
        'mobile_glb': str(mobile_glb),
        'mobile_max_faces': 34000,
        'stats_path': str(stats_path),
        'resolution': 256,
    }), encoding='utf-8')
    env = {k: v for k, v in os.environ.items() if k not in ('OPENROUTER_API_KEY', 'HF_TOKEN')}
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_idx)
    key = f'{label}/triposr'
    record = {
        'input_label': label,
        'image_source': in_entry['image_source'],
        'image_model': in_entry['model'],
        'backend': 'TripoSR',
        'appearance': 'vertex_color',
        'device': f'cuda:{gpu_idx}',
    }
    try:
        run_streaming([PYTHON, triposr_worker, cfg_path], env=env, prefix=f'triposr:{label}', timeout=600)
        stats = json.loads(stats_path.read_text(encoding='utf-8'))
        glb_meta = validate_glb(out_glb)
        mob_meta = validate_glb(mobile_glb)
        mob_stats = {**(stats.pop('mobile', None) or {}), **mob_meta}
        record.update(
            status='success',
            metrics={**stats, **glb_meta},
            mobile_metrics=mob_stats,
            mobile_glb_path=mobile_glb,
            artifact_paths=[out_glb, mobile_glb],
        )
        display(FileLink(str(out_glb)))
    except Exception as exc:
        print(f'TripoSR [{label}] failed:', str(exc)[-1200:])
        record.update(status='failed', error=type(exc).__name__, artifact_paths=[])
    finally:
        triposr_gpu_queue.put(gpu_idx)
    return key, record

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_GPU_WORKERS) as pool:
    for key, rec in pool.map(run_triposr_for, ACTIVE_LABELS):
        RUNS_3D[key] = rec
        print(key, rec['status'], rec.get('metrics', rec.get('error')))


## 3. Parallel `Hunyuan3D-2mini` reconstruction (Margin-Padded Full Plate, Raw 80k & Mobile `<=35k`)

Pads each plate-preserving RGBA cutout cleanly with `8%` margin into a square canvas before running
`Tencent/Hunyuan3D-2` (`hunyuan3d-dit-v2-mini-fast`, `fp16`, `mc_algo="mc"`), then cleans and exports
**`hunyuan3d_2mini.glb`** (`max_faces=80000`) and **`hunyuan3d_2mini_mobile.glb`** (`max_faces=34000`).


In [ ]:
import queue

HY_DIR = SCRATCH_ROOT / 'Hunyuan3D-2-src'
if not HY_DIR.exists():
    run(['git', 'init', '-q', str(HY_DIR)])
    run(['git', '-C', str(HY_DIR), 'fetch', '-q', '--depth', '1',
         'https://github.com/Tencent/Hunyuan3D-2.git', REVISIONS['hunyuan3d_repo']])
    run(['git', '-C', str(HY_DIR), 'checkout', '-q', '--detach', 'FETCH_HEAD'])

from huggingface_hub import snapshot_download
hy_snap = Path(snapshot_download(
    'tencent/Hunyuan3D-2mini',
    revision=REVISIONS['hunyuan3d_2mini_model'],
    allow_patterns=['hunyuan3d-dit-v2-mini-fast/*'],
    token=False,
))
hy_sub = hy_snap / 'hunyuan3d-dit-v2-mini-fast'
hy_ckpt = hy_sub / 'model.fp16.safetensors'
hy_cfg = hy_sub / 'config.yaml'
assert hy_ckpt.is_file() and hy_cfg.is_file(), f'Missing Hunyuan3D-2mini weights in {hy_sub}'

hy_worker_code = r'''
import json, sys, time
from pathlib import Path
import numpy as np
import torch
import trimesh
from PIL import Image

cfg = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
sys.path.insert(0, cfg['source_dir'])
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.shapegen.models.conditioner import SingleImageEncoder, DualImageEncoder
from hy3dgen.shapegen.postprocessors import FloaterRemover, DegenerateFaceRemover, FaceReducer


def remap_hunyuan_conditioner_state_dict(state_dict, target_keys):
    target_set = set(target_keys)
    uses_v5_dinov2 = any(
        k.endswith('.attention.q_proj.weight') or k.endswith('.mlp.gate_proj.weight')
        for k in target_set
    )
    if not uses_v5_dinov2:
        return state_dict

    remapped = {}
    simple_rules = (
        ('.attention.attention.query.', '.attention.q_proj.'),
        ('.attention.attention.key.', '.attention.k_proj.'),
        ('.attention.attention.value.', '.attention.v_proj.'),
        ('.attention.output.dense.', '.attention.o_proj.'),
        ('.mlp.weights_out.', '.mlp.down_proj.'),
    )
    for k, v in state_dict.items():
        if k in target_set:
            remapped[k] = v
            continue
        if '.mlp.weights_in.' in k:
            g_tensor, u_tensor = v.chunk(2, dim=0)
            gk = k.replace('.mlp.weights_in.', '.mlp.gate_proj.')
            uk = k.replace('.mlp.weights_in.', '.mlp.up_proj.')
            if gk in target_set and uk in target_set:
                remapped[gk] = g_tensor.contiguous()
                remapped[uk] = u_tensor.contiguous()
                continue
        mapped_k = k
        for old_sub, new_sub in simple_rules:
            if old_sub in mapped_k:
                mapped_k = mapped_k.replace(old_sub, new_sub)
                break
        remapped[mapped_k] = v
    return remapped


for _enc_cls in (SingleImageEncoder, DualImageEncoder):
    if not getattr(_enc_cls.load_state_dict, '_peekaplate_patched', False):
        _orig_load = _enc_cls.load_state_dict

        def _compat_load(self, state_dict, strict=True, assign=False, _orig=_orig_load):
            mapped = remap_hunyuan_conditioner_state_dict(state_dict, self.state_dict().keys())
            return _orig(self, mapped, strict=strict, assign=assign)

        _compat_load._peekaplate_patched = True
        _enc_cls.load_state_dict = _compat_load


def pad_rgba_for_reconstruction(im, margin=0.08):
    rgba = im.convert('RGBA')
    w, h = rgba.size
    longest = max(w, h)
    pad = max(1, int(round(longest * margin)))
    sq = longest + 2 * pad
    canvas = Image.new('RGBA', (sq, sq), (0, 0, 0, 0))
    canvas.paste(rgba, ((sq - w) // 2, (sq - h) // 2))
    return canvas

t0 = time.perf_counter()
print('Loading Hunyuan3D-2mini fp16 checkpoint...', flush=True)
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_single_file(
    cfg['ckpt_path'], cfg['config_path'],
    device='cuda', dtype=torch.float16, use_safetensors=True,
)
with Image.open(cfg['cutout_path']) as im:
    rgba = pad_rgba_for_reconstruction(im, margin=float(cfg.get('margin', 0.08)))

print('Running 30-step flow matching + marching cubes...', flush=True)
gen = torch.Generator('cuda').manual_seed(42)
meshes = pipe(
    image=rgba,
    num_inference_steps=30,
    guidance_scale=5.0,
    octree_resolution=int(cfg['octree_resolution']),
    mc_algo='mc',
    num_chunks=8000,
    generator=gen,
    enable_pbar=False,
)
mesh = meshes[0] if isinstance(meshes, list) else meshes
if mesh is None or not isinstance(mesh, trimesh.Trimesh) or len(mesh.faces) == 0:
    raise ValueError('Hunyuan3D-2mini returned an empty mesh.')
if not np.isfinite(mesh.vertices).all():
    raise ValueError('Hunyuan3D-2mini returned non-finite vertices.')

raw_vertices = int(len(mesh.vertices))
raw_faces = int(len(mesh.faces))
max_faces = int(cfg.get('max_faces', 80000))
floater_remover = FloaterRemover()
degen_remover = DegenerateFaceRemover()
face_reducer = FaceReducer()

for step_name, fn in (
    ('FloaterRemover', floater_remover),
    ('DegenerateFaceRemover', degen_remover),
    ('FaceReducer', lambda m: face_reducer(m, max_facenum=max_faces)),
):
    candidate = fn(mesh)
    if candidate is None or not isinstance(candidate, trimesh.Trimesh) or len(candidate.faces) == 0:
        raise ValueError(f'Hunyuan3D-2mini {step_name} produced an empty mesh.')
    if not np.isfinite(candidate.vertices).all():
        raise ValueError(f'Hunyuan3D-2mini {step_name} produced non-finite vertices.')
    mesh = candidate

out_glb = Path(cfg['output_glb'])
out_glb.parent.mkdir(parents=True, exist_ok=True)
out_glb.write_bytes(mesh.export(file_type='glb'))

mobile_stats = None
if cfg.get('mobile_glb'):
    mobile_glb = Path(cfg['mobile_glb'])
    mob_target = int(cfg.get('mobile_max_faces', 34000))
    mob_mesh = face_reducer(mesh, max_facenum=mob_target)
    if mob_mesh is None or not isinstance(mob_mesh, trimesh.Trimesh) or len(mob_mesh.faces) == 0:
        raise ValueError('Hunyuan3D-2mini mobile FaceReducer produced an empty mesh.')
    if not np.isfinite(mob_mesh.vertices).all():
        raise ValueError('Hunyuan3D-2mini mobile FaceReducer produced non-finite vertices.')
    mobile_glb.parent.mkdir(parents=True, exist_ok=True)
    mobile_glb.write_bytes(mob_mesh.export(file_type='glb'))
    mobile_stats = {'vertices': int(len(mob_mesh.vertices)), 'faces': int(len(mob_mesh.faces))}

bounds = mesh.bounds.tolist() if mesh.bounds is not None else None
stats = {
    'raw_vertices': raw_vertices,
    'raw_faces': raw_faces,
    'vertices': int(len(mesh.vertices)),
    'faces': int(len(mesh.faces)),
    'max_faces': max_faces,
    'mobile': mobile_stats,
    'margin': float(cfg.get('margin', 0.08)),
    'bounds': bounds,
    'octree_resolution': int(cfg.get('octree_resolution')),
    'subfolder': 'hunyuan3d-dit-v2-mini-fast',
    'elapsed_seconds': round(time.perf_counter() - t0, 2),
    'gpu_name': torch.cuda.get_device_name(0),
    'peak_vram_gib': round(torch.cuda.max_memory_allocated(0) / 2**30, 3),
}
Path(cfg['stats_path']).write_text(json.dumps(stats), encoding='utf-8')
print(
    f"Hunyuan3D-2mini exported v={stats['vertices']} (raw {raw_vertices}) "
    f"f={stats['faces']} (raw {raw_faces}, mobile {mobile_stats['faces'] if mobile_stats else '-'}) in {stats['elapsed_seconds']}s",
    flush=True,
)
'''
hy_worker = SCRATCH_ROOT / 'run_hunyuan_worker.py'
hy_worker.write_text(hy_worker_code, encoding='utf-8')

hunyuan_gpu_queue = queue.Queue()
for gid in range(MAX_GPU_WORKERS):
    hunyuan_gpu_queue.put(gid)

def run_hunyuan_for(label):
    gpu_idx = hunyuan_gpu_queue.get()
    in_entry = image_results[label]
    out_glb = RESULTS_DIR / label / 'hunyuan3d_2mini.glb'
    mobile_glb = RESULTS_DIR / label / 'hunyuan3d_2mini_mobile.glb'
    stats_path = SCRATCH_ROOT / f'hunyuan_stats_{label}.json'
    cfg_path = SCRATCH_ROOT / f'hunyuan_cfg_{label}.json'
    cfg_path.write_text(json.dumps({
        'source_dir': str(HY_DIR),
        'ckpt_path': str(hy_ckpt),
        'config_path': str(hy_cfg),
        'cutout_path': str(in_entry['cutout_path']),
        'output_glb': str(out_glb),
        'mobile_glb': str(mobile_glb),
        'mobile_max_faces': 34000,
        'margin': 0.08,
        'stats_path': str(stats_path),
        'octree_resolution': 256,
        'max_faces': 80000,
    }), encoding='utf-8')
    env = {k: v for k, v in os.environ.items() if k not in ('OPENROUTER_API_KEY', 'HF_TOKEN')}
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_idx)
    key = f'{label}/hunyuan3d-2mini'
    record = {
        'input_label': label,
        'image_source': in_entry['image_source'],
        'image_model': in_entry['model'],
        'backend': 'Hunyuan3D-2mini',
        'appearance': 'geometry_only',
        'device': f'cuda:{gpu_idx}',
    }
    try:
        run_streaming([PYTHON, hy_worker, cfg_path], env=env, prefix=f'hunyuan:{label}', timeout=600)
        stats = json.loads(stats_path.read_text(encoding='utf-8'))
        glb_meta = validate_glb(out_glb)
        mob_meta = validate_glb(mobile_glb)
        mob_stats = {**(stats.pop('mobile', None) or {}), **mob_meta}
        record.update(
            status='success',
            metrics={**stats, **glb_meta},
            mobile_metrics=mob_stats,
            mobile_glb_path=mobile_glb,
            artifact_paths=[out_glb, mobile_glb],
        )
        display(FileLink(str(out_glb)))
    except Exception as exc:
        print(f'Hunyuan3D-2mini [{label}] failed:', str(exc)[-1200:])
        record.update(status='failed', error=type(exc).__name__, artifact_paths=[])
    finally:
        hunyuan_gpu_queue.put(gpu_idx)
    return key, record

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_GPU_WORKERS) as pool:
    for key, rec in pool.map(run_hunyuan_for, ACTIVE_LABELS):
        RUNS_3D[key] = rec
        print(key, rec['status'], rec.get('metrics', rec.get('error')))


## 4. Experimental `TRELLIS.2-4B` native setup (gated on `DINOV3_OK`)

This section builds `CuMesh`, `O-Voxel`, `FlexGEMM`, and `nvdiffrast` for compute capability 7.5 **only when
`HF_TOKEN` passes the upfront DINOv3 preflight check** and `nvcc` is available (`Release 12.x`).
- Broken `flex_gemm` Python-only fallback has been removed (native build is required if `DINOV3_OK` is true).
- `nvdiffrec_render` (used only for optional 6-mode HF Space preview shaders, not GLB export) is treated as optional
  so a `nvdiffrec_render` wheel error never blocks `TRELLIS.2-4B` GLB generation.


In [ ]:
TRELLIS_SOURCE = SCRATCH_ROOT / 'TRELLIS.2-native'
TRELLIS_REVISIONS = {
    'trellis': '75fbf0183001ed9876c8dbb35de6b68552ee08bd',
    'flex_gemm': '6dd94a859c26ee8246888502eada3dd8ad85532e',
    'cumesh': '12289e1062f0603f2f0d0771b02e1395d247f26f',
    'nvdiffrast': '253ac4fcea7de5f396371124af597e6cc957bfae',
    'nvdiffrec': 'b296927cc7fd01c2ac1087c8065c4d7248f72da4',
}
REVISIONS.update(TRELLIS_REVISIONS)
BUILD_ENV = os.environ.copy()
for secret_name in ('OPENROUTER_API_KEY', 'HF_TOKEN'):
    BUILD_ENV.pop(secret_name, None)
cuda_caps = sorted({torch.cuda.get_device_capability(idx) for idx in range(GPU_COUNT)})
BUILD_ENV['TORCH_CUDA_ARCH_LIST'] = ';'.join(f'{maj}.{min_}' for maj, min_ in cuda_caps)
BUILD_ENV['MAX_JOBS'] = '2'
if Path('/usr/local/cuda').exists():
    BUILD_ENV['CUDA_HOME'] = '/usr/local/cuda'

NVDIFFREC_AVAILABLE = False
TRELLIS_BUILD_OK = False
TRELLIS_SKIP_REASON = None

if not DINOV3_OK:
    TRELLIS_SKIP_REASON = DINOV3_NOTE
    print(f'Skipping TRELLIS.2 native CUDA build: {DINOV3_NOTE}. Attach an authorized HF_TOKEN in Kaggle Secrets to enable TRELLIS.2-4B.')
elif not NVCC_AVAILABLE:
    TRELLIS_SKIP_REASON = 'nvcc_unavailable'
    print('Skipping TRELLIS.2 native CUDA build: nvcc compiler unavailable.')
elif len(cuda_caps) != 1:
    TRELLIS_SKIP_REASON = f'heterogeneous_cuda_caps_{cuda_caps}'
    print(f'Skipping TRELLIS.2 native CUDA build: heterogeneous CUDA compute capabilities detected: {cuda_caps}')
elif ('release 12.' not in subprocess.check_output(['nvcc', '--version'], text=True)) or (not str(torch.version.cuda).startswith('12.')):
    TRELLIS_SKIP_REASON = 'requires_cuda_12_toolchain'
    print(f'Skipping TRELLIS.2 native CUDA build: FlexGEMM/CuMesh/nvdiffrast pin CUDA 12.x (detected CUDA {torch.version.cuda}); continuing with TripoSR + Hunyuan3D-2mini.')
else:

    run([PYTHON, '-m', 'pip', 'install',
         'wrapt', 'setuptools<81', 'wheel', 'ninja==1.11.1.4',
         'huggingface_hub==0.36.0', 'safetensors==0.6.2', 'transformers==4.57.3',
         'imageio==2.37.2', 'imageio-ffmpeg==0.6.0', 'tqdm==4.67.1', 'easydict==1.13',
         'opencv-python-headless==4.12.0.88', 'trimesh==4.10.1',
         'kornia==0.8.2', 'timm==1.0.22', 'zstandard==0.25.0',
         'rembg[cpu]==2.0.67', 'pyrender==0.1.45',
         'git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8'],
        env=BUILD_ENV)

    def checkout(repo, folder, rev):
        created = not folder.exists()
        if created:
            run(['git', 'clone', '--no-checkout', repo, folder])
        current = subprocess.check_output(['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True).strip()
        if created or not (folder / '.git' / 'index').exists() or current != rev:
            run(['git', '-C', folder, 'fetch', '--depth', '1', 'origin', rev])
            run(['git', '-C', folder, 'checkout', '--detach', rev])
        run(['git', '-C', folder, 'submodule', 'update', '--init', '--recursive'])

    checkout('https://github.com/microsoft/TRELLIS.2.git', TRELLIS_SOURCE, TRELLIS_REVISIONS['trellis'])
    dependencies = SCRATCH_ROOT / 'trellis_native_dependencies'
    dependencies.mkdir(exist_ok=True)

    for name, repository in (
        ('flex_gemm', 'https://github.com/JeffreyXiang/FlexGEMM.git'),
        ('cumesh', 'https://github.com/JeffreyXiang/CuMesh.git'),
        ('nvdiffrast', 'https://github.com/NVlabs/nvdiffrast.git'),
        ('nvdiffrec', 'https://github.com/JeffreyXiang/nvdiffrec.git'),
    ):
        directory = dependencies / name
        checkout(repository, directory, TRELLIS_REVISIONS[name])
        try:
            run([PYTHON, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', str(directory)], env=BUILD_ENV)
            if name == 'nvdiffrec':
                NVDIFFREC_AVAILABLE = True
        except Exception as err:
            if name == 'nvdiffrec':
                print('Optional nvdiffrec_render build failed; continuing GLB export without HF Space shader previews:', err)
                NVDIFFREC_AVAILABLE = False
            else:
                raise

    run([PYTHON, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', TRELLIS_SOURCE / 'o-voxel'], env=BUILD_ENV)
    TRELLIS_BUILD_OK = True
    print('TRELLIS.2 native build complete. nvdiffrec_available =', NVDIFFREC_AVAILABLE)


## 5. Write and check the TRELLIS.2 T4 adapters (always unit-tested)

Writes `trellis_native_compat.py` (with `points.trunc().long()` nearest-sampling parity fix),
`trellis_native_runner.py` (with `pinned = config.get('checkpoint_revisions') or {}` fix),
`trellis_prepare_full_plate.py`, `trellis_native_space_tools.py`, and their unit suites, then runs
all CPU/GPU adapter checks.


In [ ]:
(SCRATCH_ROOT / 'trellis_native_compat.py').write_text('"""Experimental, inference-only PyTorch adapters for official TRELLIS.2 weights.\n\nThe submanifold-convolution parameter names and layout match Microsoft\'s\nFlexGEMM adapter: [out_channels, kernel_x, kernel_y, kernel_z, in_channels].\nThese operations prioritize T4 compatibility over throughput.\n"""\n\nimport importlib\nimport itertools\nimport math\nfrom types import SimpleNamespace\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass CoordinateLookup:\n    def __init__(self, coords, spatial_shape=None):\n        if coords.ndim != 2 or coords.shape[1] != 4 or len(coords) == 0:\n            raise ValueError(\'Expected nonempty [N, 4] batch/xyz coordinates.\')\n        self.coords = coords.long()\n        self.spatial_shape = tuple(int(v) for v in (\n            spatial_shape if spatial_shape is not None else self.coords[:, 1:].amax(0) + 1\n        ))\n        self.batch_size = int(self.coords[:, 0].max()) + 1\n        self.keys, self.order = self.encode(self.coords).sort()\n        if len(self.keys) > 1 and bool((self.keys[1:] == self.keys[:-1]).any()):\n            raise ValueError(\'Duplicate sparse coordinates are unsupported.\')\n\n    def encode(self, coords):\n        width, height, depth = self.spatial_shape\n        return ((coords[:, 0] * width + coords[:, 1]) * height + coords[:, 2]) * depth + coords[:, 3]\n\n    def find(self, query):\n        query = query.long()\n        limits = query.new_tensor((self.batch_size, *self.spatial_shape))\n        valid = ((query >= 0) & (query < limits)).all(1)\n        keys = self.encode(query)\n        positions = torch.searchsorted(self.keys, keys).clamp_max(len(self.keys) - 1)\n        valid = valid & (self.keys[positions] == keys)\n        return torch.where(valid, self.order[positions], positions.new_full(positions.shape, -1))\n\n\ndef submanifold_convolution(feats, coords, weight, bias=None, dilation=(1, 1, 1),\n                            neighbor_map=None, chunk_size=8192):\n    # 1. Keep exactly the checkpoint\'s convolution orientation and sparse sites.\n    out_channels, size_x, size_y, size_z, in_channels = weight.shape\n    if feats.shape != (len(coords), in_channels):\n        raise ValueError(\'Sparse feature channels do not match convolution weights.\')\n    kernel_shape = (size_x, size_y, size_z)\n    if any(size % 2 == 0 for size in kernel_shape):\n        raise ValueError(\'This adapter supports odd, centered submanifold kernels.\')\n    offsets = list(itertools.product(*(range(size) for size in kernel_shape)))\n    if neighbor_map is None:\n        lookup = CoordinateLookup(coords)\n        neighbor_map = torch.empty((len(coords), len(offsets)), dtype=torch.int32, device=coords.device)\n        for index, offset in enumerate(offsets):\n            query = coords.clone().long()\n            delta = query.new_tensor([(offset[axis] - kernel_shape[axis] // 2) * dilation[axis]\n                                      for axis in range(3)])\n            query[:, 1:] += delta\n            neighbor_map[:, index] = lookup.find(query).int()\n\n    # 2. Bound temporary activation memory and accumulate convolution in FP32.\n    output = torch.empty((len(coords), out_channels), dtype=feats.dtype, device=feats.device)\n    for start in range(0, len(coords), chunk_size):\n        stop = min(start + chunk_size, len(coords))\n        accumulator = torch.zeros((stop - start, out_channels), dtype=torch.float32, device=feats.device)\n        for index, (ix, iy, iz) in enumerate(offsets):\n            neighbors = neighbor_map[start:stop, index].long()\n            valid = neighbors >= 0\n            gathered = feats[neighbors.clamp_min(0)].float()\n            gathered = gathered * valid[:, None]\n            accumulator.addmm_(gathered, weight[:, ix, iy, iz, :].float().t())\n        if bias is not None:\n            accumulator += bias.float()\n        output[start:stop] = accumulator.to(feats.dtype)\n    return output, neighbor_map\n\n\ndef sparse_grid_sample(feats, coords, shape, grid, mode=\'trilinear\', chunk_size=32768):\n    # 3. Sample voxel-centered attributes without the Triton interpolation kernel.\n    if mode not in (\'nearest\', \'trilinear\'):\n        raise ValueError(f\'Unknown sampling mode: {mode}\')\n    if grid.ndim != 3 or grid.shape[-1] != 3:\n        raise ValueError(\'Expected grid shape [batch, query_count, 3].\')\n    lookup = CoordinateLookup(coords, shape[-3:])\n    batch_size, query_count, _ = grid.shape\n    batches = torch.arange(batch_size, device=grid.device).repeat_interleave(query_count)\n    queries = grid.reshape(-1, 3).float()\n    output = torch.empty((len(queries), feats.shape[1]), dtype=feats.dtype, device=feats.device)\n    for start in range(0, len(queries), chunk_size):\n        stop = min(start + chunk_size, len(queries))\n        points = queries[start:stop]\n        batch_column = batches[start:stop, None]\n        if mode == \'nearest\':\n            indices = lookup.find(torch.cat([batch_column, points.trunc().long()], 1))\n            output[start:stop] = feats[indices.clamp_min(0)] * (indices >= 0)[:, None]\n            continue\n        lower = (points - 0.5).floor().long()\n        value_sum = torch.zeros((len(points), feats.shape[1]), dtype=torch.float32, device=feats.device)\n        weight_sum = torch.zeros((len(points), 1), dtype=torch.float32, device=feats.device)\n        for offset in itertools.product((0, 1), repeat=3):\n            neighbor = lower + lower.new_tensor(offset)\n            indices = lookup.find(torch.cat([batch_column, neighbor], 1))\n            weights = (1 - (neighbor.float() + 0.5 - points).abs()).clamp_min(0).prod(1, keepdim=True)\n            weights *= (indices >= 0)[:, None]\n            value_sum += weights * feats[indices.clamp_min(0)].float()\n            weight_sum += weights\n        output[start:stop] = (value_sum / weight_sum.clamp_min(1e-12)).to(feats.dtype)\n    return output.reshape(batch_size, query_count, -1)\n\n\ndef sparse_attention(*args, **kwargs):\n    # 4. Preserve each batch\'s independent sequence and the public return type.\n    names = {1: (\'qkv\',), 2: (\'q\', \'kv\'), 3: (\'q\', \'k\', \'v\')}\n    count = len(args) + len(kwargs)\n    if count not in names:\n        raise ValueError(\'Expected qkv, q/kv, or q/k/v.\')\n    values = list(args) + [kwargs[name] for name in names[count][len(args):]]\n\n    def unpack(value):\n        if hasattr(value, \'feats\'):\n            return value.feats, value.layout\n        batch_size, length = value.shape[:2]\n        return value.flatten(0, 1), [slice(i * length, (i + 1) * length) for i in range(batch_size)]\n\n    source = values[0] if hasattr(values[0], \'feats\') else None\n    if count == 1:\n        tensor, q_layout = unpack(values[0])\n        query, key, value = tensor.unbind(1)\n        kv_layout = q_layout\n    elif count == 2:\n        query, q_layout = unpack(values[0])\n        tensor, kv_layout = unpack(values[1])\n        key, value = tensor.unbind(1)\n    else:\n        query, q_layout = unpack(values[0])\n        key, kv_layout = unpack(values[1])\n        value, value_layout = unpack(values[2])\n        if [(s.start, s.stop) for s in value_layout] != [(s.start, s.stop) for s in kv_layout]:\n            raise ValueError(\'Key and value batch layouts differ.\')\n    if len(q_layout) != len(kv_layout):\n        raise ValueError(\'Query and key/value batch sizes differ.\')\n    output = query.new_empty((len(query), query.shape[1], value.shape[-1]))\n    for query_slice, key_slice in zip(q_layout, kv_layout):\n        keys = key[key_slice].transpose(0, 1).unsqueeze(0)\n        vals = value[key_slice].transpose(0, 1).unsqueeze(0)\n        for start in range(query_slice.start, query_slice.stop, 256):\n            stop = min(start + 256, query_slice.stop)\n            queries = query[start:stop].transpose(0, 1).unsqueeze(0)\n            result = F.scaled_dot_product_attention(queries, keys, vals, dropout_p=0.0)\n            output[start:stop] = result.squeeze(0).transpose(0, 1)\n    if source is not None:\n        return source.replace(output)\n    return output.reshape(values[0].shape[0], values[0].shape[1], *output.shape[1:])\n\n\ndef install_adapters():\n    # 5. Register a weight-compatible backend before constructing any decoder.\n    from trellis2.modules import sparse as sp\n    from trellis2.modules.sparse import config\n    config.set_conv_backend(\'none\')\n    conv_module = importlib.import_module(\'trellis2.modules.sparse.conv.conv\')\n\n    def initialize(module, in_channels, out_channels, kernel_size, stride=1,\n                   dilation=1, padding=None, bias=True, indice_key=None):\n        if stride != 1 or padding is not None:\n            raise NotImplementedError(\'Only the official submanifold-convolution path is supported.\')\n        sizes = tuple(kernel_size) if isinstance(kernel_size, (tuple, list)) else (kernel_size,) * 3\n        module.dilation = tuple(dilation) if isinstance(dilation, (tuple, list)) else (dilation,) * 3\n        module.in_channels, module.out_channels = in_channels, out_channels\n        module.weight = nn.Parameter(torch.empty(out_channels, *sizes, in_channels))\n        nn.init.kaiming_uniform_(module.weight, a=math.sqrt(5))\n        if bias:\n            module.bias = nn.Parameter(torch.zeros(out_channels))\n        else:\n            module.register_parameter(\'bias\', None)\n\n    def forward(module, value):\n        key = (\'native_torch_neighbors\', tuple(module.weight.shape[1:4]), module.dilation)\n        cached = value.get_spatial_cache(key)\n        features, neighbors = submanifold_convolution(\n            value.feats, value.coords, module.weight, module.bias, module.dilation, cached\n        )\n        if cached is None:\n            value.register_spatial_cache(key, neighbors)\n        return value.replace(features)\n\n    def unsupported(*args, **kwargs):\n        raise NotImplementedError(\'Inverse sparse convolution is not part of this inference adapter.\')\n\n    conv_module._backends[\'none\'] = SimpleNamespace(\n        sparse_conv3d_init=initialize, sparse_conv3d_forward=forward,\n        sparse_inverse_conv3d_init=unsupported, sparse_inverse_conv3d_forward=unsupported,\n    )\n    for name in (\'trellis2.modules.sparse.attention\',\n                 \'trellis2.modules.sparse.attention.full_attn\',\n                 \'trellis2.modules.sparse.attention.modules\'):\n        module = importlib.import_module(name)\n        module.sparse_scaled_dot_product_attention = sparse_attention\n    sp.sparse_scaled_dot_product_attention = sparse_attention\n    mesh_module = importlib.import_module(\'trellis2.representations.mesh.base\')\n    mesh_module.grid_sample_3d = sparse_grid_sample\n    export_module = importlib.import_module(\'o_voxel.postprocess\')\n    export_module.grid_sample_3d = sparse_grid_sample\n', encoding='utf-8')
(SCRATCH_ROOT / 'trellis_native_runner.py').write_text('"""Direct Microsoft TRELLIS.2-4B inference; experimental Kaggle T4 configuration."""\n\nimport argparse\nimport gc\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport traceback\nimport types\n\n# 1. Select supported attention before importing TRELLIS modules.\nos.environ[\'OPENCV_IO_ENABLE_OPENEXR\'] = \'1\'\nos.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\nos.environ[\'ATTN_BACKEND\'] = \'sdpa\'\nos.environ[\'SPARSE_CONV_BACKEND\'] = \'none\'\nos.environ[\'SPARSE_ATTN_BACKEND\'] = \'xformers\'  # Replaced by our native SDPA adapter before inference.\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom huggingface_hub import HfApi, hf_hub_download, snapshot_download\nfrom safetensors.torch import load_file\n\n\ndef write_json(path, value):\n    path.write_text(json.dumps(value, indent=2), encoding=\'utf-8\')\n\n\ndef main(config_path):\n    config = json.loads(Path(config_path).read_text(encoding=\'utf-8\'))\n    output = Path(config[\'output_dir\'])\n    output.mkdir(parents=True, exist_ok=True)\n    status_path = output / \'run_status.json\'\n    status = {\'status\': \'running\', \'started_at\': time.time(), \'model\': \'microsoft/TRELLIS.2-4B\'}\n    write_json(status_path, status)\n    try:\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'A CUDA GPU is required for model decoding and GLB export.\')\n        torch.cuda.set_device(0)\n        sys.path.insert(0, config[\'source_dir\'])\n        from trellis_native_compat import install_adapters\n        install_adapters()\n        from trellis2 import models\n        from trellis2.modules import image_feature_extractor\n        from trellis2.pipelines import Trellis2ImageTo3DPipeline, samplers\n        import o_voxel\n\n        # 2. Pin the model and every external model reference used in this run.\n        api = HfApi()\n        token = os.environ.get(\'HF_TOKEN\') or None\n        pinned = config.get(\'checkpoint_revisions\') or {}\n        model_repo = \'microsoft/TRELLIS.2-4B\'\n        revision = config.get(\'model_revision\') or pinned.get(model_repo) or api.model_info(model_repo, token=token).sha\n        pipeline_path = hf_hub_download(model_repo, \'pipeline.json\', revision=revision, token=token)\n        specification = json.loads(Path(pipeline_path).read_text(encoding=\'utf-8\'))[\'args\']\n        revisions = {model_repo: revision}\n        required = (\n            \'sparse_structure_flow_model\', \'sparse_structure_decoder\',\n            \'shape_slat_flow_model_512\', \'shape_slat_decoder\',\n            \'tex_slat_flow_model_512\', \'tex_slat_decoder\',\n        )\n        loaded_models = {}\n        checkpoint_audit = {}\n        allowed_generated_buffers = (\'pos_emb\', \'rope_phases\')\n\n        for role in required:\n            reference = specification[\'models\'][role]\n            if reference.startswith(\'ckpts/\'):\n                repository, prefix = model_repo, reference\n            else:\n                owner, name, prefix = reference.split(\'/\', 2)\n                repository = owner + \'/\' + name\n            if repository not in revisions:\n                revisions[repository] = pinned.get(repository) or api.model_info(repository, token=token).sha\n            model_json = hf_hub_download(repository, prefix + \'.json\', revision=revisions[repository], token=token)\n            model_weights = hf_hub_download(repository, prefix + \'.safetensors\', revision=revisions[repository], token=token)\n            model_spec = json.loads(Path(model_json).read_text(encoding=\'utf-8\'))\n            arguments = model_spec[\'args\'].copy()\n            if \'flow_model\' in role:\n                # T4 cannot execute the original BF16 transformer path natively.\n                arguments[\'dtype\'] = \'float32\'\n            print(\'Loading official checkpoint:\', role, flush=True)\n            model = getattr(models, model_spec[\'name\'])(**arguments).eval()\n            weights = load_file(model_weights, device=\'cpu\')\n            incompatible = model.load_state_dict(weights, strict=False)\n            missing = [key for key in incompatible.missing_keys\n                       if key.rsplit(\'.\', 1)[-1] not in allowed_generated_buffers]\n            if missing or incompatible.unexpected_keys:\n                raise RuntimeError(f\'Checkpoint mismatch for {role}: missing={missing}; \'\n                                   f\'unexpected={incompatible.unexpected_keys}\')\n            checkpoint_audit[role] = {\n                \'repository\': repository, \'revision\': revisions[repository], \'prefix\': prefix,\n                \'generated_buffers_not_stored\': list(incompatible.missing_keys),\n            }\n            if \'flow_model\' in role:\n                model.convert_to(torch.float32)\n            loaded_models[role] = model.cpu()\n            del weights\n            gc.collect()\n\n        # 3. Load the configured DINOv3 encoder; an access-approved HF token may be required.\n        encoder_arguments = specification[\'image_cond_model\'][\'args\'].copy()\n        encoder_repo = encoder_arguments[\'model_name\']\n        revisions[encoder_repo] = pinned.get(encoder_repo) or api.model_info(encoder_repo, token=token).sha\n        encoder_directory = snapshot_download(\n            encoder_repo, revision=revisions[encoder_repo], token=token,\n            allow_patterns=[\'config.json\', \'*.safetensors\', \'*.safetensors.index.json\'],\n        )\n        encoder_arguments[\'model_name\'] = encoder_directory\n        encoder_class = getattr(image_feature_extractor, specification[\'image_cond_model\'][\'name\'])\n        encoder = encoder_class(**encoder_arguments)\n        encoder.model.float().eval()\n\n        def make_sampler(name):\n            entry = specification[name]\n            return getattr(samplers, entry[\'name\'])(**entry[\'args\'])\n\n        pipeline = Trellis2ImageTo3DPipeline(\n            models=loaded_models,\n            sparse_structure_sampler=make_sampler(\'sparse_structure_sampler\'),\n            shape_slat_sampler=make_sampler(\'shape_slat_sampler\'),\n            tex_slat_sampler=make_sampler(\'tex_slat_sampler\'),\n            sparse_structure_sampler_params=specification[\'sparse_structure_sampler\'][\'params\'],\n            shape_slat_sampler_params=specification[\'shape_slat_sampler\'][\'params\'],\n            tex_slat_sampler_params=specification[\'tex_slat_sampler\'][\'params\'],\n            shape_slat_normalization=specification[\'shape_slat_normalization\'],\n            tex_slat_normalization=specification[\'tex_slat_normalization\'],\n            image_cond_model=encoder, rembg_model=None, low_vram=True,\n            default_pipeline_type=\'512\',\n        )\n        pipeline.to(torch.device(\'cuda:0\'))\n        encoder_device = 1 if torch.cuda.device_count() > 1 else 0\n\n        @torch.no_grad()\n        def get_condition(self, image, resolution, include_neg_cond=True):\n            self.image_cond_model.image_size = resolution\n            with torch.cuda.device(encoder_device):\n                self.image_cond_model.to(torch.device(f\'cuda:{encoder_device}\'))\n                try:\n                    features = self.image_cond_model(image).float()\n                finally:\n                    self.image_cond_model.cpu()\n            features = features.to(self.device)\n            if not torch.isfinite(features).all():\n                raise RuntimeError(\'DINO image features contain NaN/Inf.\')\n            return {\'cond\': features, \'neg_cond\': torch.zeros_like(features)} if include_neg_cond else {\'cond\': features}\n\n        pipeline.get_cond = types.MethodType(get_condition, pipeline)\n        write_json(output / \'model_revision.json\', {\'revisions\': revisions, \'checkpoints\': checkpoint_audit})\n\n        # 4. Run the official 512 path with sequential CPU/GPU offloading.\n        torch.cuda.reset_peak_memory_stats(0)\n        image = Image.open(config[\'prepared_image\']).convert(\'RGB\')\n        started = time.perf_counter()\n        meshes, latents = pipeline.run(\n            image, seed=config[\'seed\'], pipeline_type=\'512\', preprocess_image=False,\n            sparse_structure_sampler_params=config.get(\'samplers\', {}).get(\'sparse_structure\', {\'steps\': config[\'sampling_steps\']}),\n            shape_slat_sampler_params=config.get(\'samplers\', {}).get(\'shape\', {\'steps\': config[\'sampling_steps\']}),\n            tex_slat_sampler_params=config.get(\'samplers\', {}).get(\'material\', {\'steps\': config[\'sampling_steps\']}),\n            return_latent=True,\n        )\n        mesh = meshes[0]\n        for label, tensor in ((\'vertices\', mesh.vertices), (\'attributes\', mesh.attrs),\n                              (\'shape latent\', latents[0].feats), (\'material latent\', latents[1].feats)):\n            if not torch.isfinite(tensor).all():\n                raise RuntimeError(f\'Generated {label} contains NaN/Inf; no result will be declared successful.\')\n        if len(mesh.vertices) == 0 or len(mesh.faces) == 0:\n            raise RuntimeError(\'The model generated an empty mesh.\')\n        generation_seconds = time.perf_counter() - started\n        np.savez_compressed(output / \'raw_mesh_and_materials.npz\',\n                            vertices=mesh.vertices.cpu().numpy(), faces=mesh.faces.cpu().numpy(),\n                            coords=mesh.coords.cpu().numpy(), attributes=mesh.attrs.float().cpu().numpy(),\n                            voxel_size=np.array(mesh.voxel_size), origin=mesh.origin.cpu().numpy(),\n                            voxel_shape=np.array(list(mesh.voxel_shape)))\n        layout_dict = {k: [v.start, v.stop, v.step] for k, v in mesh.layout.items()}\n        metadata = {\n            \'attribute_layout\': layout_dict,\n            \'voxel_size\': float(mesh.voxel_size.item() if hasattr(mesh.voxel_size, \'item\') else mesh.voxel_size),\n            \'voxel_shape\': list(mesh.voxel_shape),\n            \'origin\': mesh.origin.cpu().tolist() if hasattr(mesh.origin, \'cpu\') else list(mesh.origin),\n        }\n        (output / \'raw_mesh_metadata.json\').write_text(json.dumps(metadata, indent=2), encoding=\'utf-8\')\n        if config.get(\'save_latents\', False):\n            torch.save({\'shape_features\': latents[0].feats.cpu(), \'material_features\': latents[1].feats.cpu(),\n                        \'coords\': latents[0].coords.cpu(), \'resolution\': latents[2]}, output / \'generated_latents.pt\')\n        del latents\n        for model in pipeline.models.values():\n            model.cpu()\n        gc.collect()\n        torch.cuda.empty_cache()\n\n        # 5. Export actual textured geometry, then verify that a nonempty GLB exists.\n        started = time.perf_counter()\n        asset = o_voxel.postprocess.to_glb(\n            vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,\n            coords=mesh.coords, attr_layout=mesh.layout, voxel_size=mesh.voxel_size,\n            aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],\n            decimation_target=config[\'decimation_target\'], texture_size=config[\'texture_size\'],\n            remesh=config[\'remesh\'], remesh_band=1, remesh_project=0,\n            use_tqdm=True,\n        )\n        target = output / \'generated_original.glb\'\n        asset.export(str(target), extension_webp=True)\n        with target.open(\'rb\') as stream:\n            if stream.read(4) != b\'glTF\' or target.stat().st_size <= 20:\n                raise RuntimeError(\'Export did not create a valid GLB container.\')\n        status.update(status=\'success\', generation_seconds=generation_seconds,\n                      export_seconds=time.perf_counter() - started, output=str(target),\n                      peak_gpu0_gib=torch.cuda.max_memory_allocated(0) / 2**30,\n                      encoder_gpu=encoder_device, ended_at=time.time())\n        write_json(status_path, status)\n        print(\'Saved:\', target, flush=True)\n    except Exception as error:\n        status.update(status=\'failed\', error_type=type(error).__name__, ended_at=time.time())\n        write_json(status_path, status)\n        traceback.print_exc()\n        raise\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    main(parser.parse_args().config)\n', encoding='utf-8')
(SCRATCH_ROOT / 'trellis_prepare_full_plate.py').write_text('"""Preserve the complete photograph and pad it for image-to-3D conditioning."""\n\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\n\n\ndef prepare_full_plate(source, destination, margin=0.08, max_size=1024, reviewed_mask=None):\n    # 1. Retain the entire image, including the plate rim and food.\n    if not 0 <= margin <= 0.5 or max_size < 32:\n        raise ValueError(\'Use a margin from 0 to 0.5 and an image size of at least 32 pixels.\')\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    with Image.open(source) as opened:\n        image = ImageOps.exif_transpose(opened).convert(\'RGBA\')\n    original_size = image.size\n    image.save(destination / \'original_input.png\')\n\n    # An optional hand-reviewed mask must include BOTH the complete plate and the food.\n    # No automatic background remover or foreground bounding-box crop is used.\n    if reviewed_mask:\n        with Image.open(reviewed_mask) as opened:\n            mask = ImageOps.exif_transpose(opened).convert(\'L\')\n        if mask.size != image.size:\n            raise ValueError(\'The reviewed full-plate mask must match the original image dimensions.\')\n        alpha = np.asarray(image.getchannel(\'A\'), dtype=np.uint16)\n        alpha = ((alpha * np.asarray(mask, dtype=np.uint16) + 127) // 255).astype(np.uint8)\n        image.putalpha(Image.fromarray(alpha))\n    if image.getchannel(\'A\').getextrema()[1] == 0:\n        raise ValueError(\'The source or reviewed mask is fully transparent. Supply the original full plate image.\')\n\n    # 2. Fit every input pixel inside a padded square; never crop to the food silhouette.\n    longest_side = max(image.size)\n    border = math.ceil(longest_side * margin)\n    square_size = longest_side + 2 * border\n    scale = min(1, max_size / square_size)\n    resized_size = tuple(max(1, round(value * scale)) for value in image.size)\n    square_size = min(square_size, max_size)\n    if resized_size != image.size:\n        image = image.resize(resized_size, Image.Resampling.LANCZOS)\n    left = (square_size - image.width) // 2\n    top = (square_size - image.height) // 2\n    square = Image.new(\'RGBA\', (square_size, square_size), (0, 0, 0, 0))\n    square.paste(image, (left, top))\n    square.save(destination / \'full_plate_rgba.png\')\n\n    # 3. Composite existing transparency on black, as expected by the image encoder.\n    prepared = Image.alpha_composite(Image.new(\'RGBA\', square.size, (0, 0, 0, 255)), square).convert(\'RGB\')\n    prepared.save(destination / \'prepared_image.png\')\n    report = {\n        \'source\': str(source), \'original_size\': list(original_size),\n        \'prepared_size\': list(prepared.size), \'resized_image_size\': list(resized_size),\n        \'image_offset\': [left, top], \'margin\': margin,\n        \'automatic_background_removal\': False, \'foreground_cropped\': False,\n        \'reviewed_full_plate_mask\': str(reviewed_mask) if reviewed_mask else None,\n        \'existing_alpha_preserved\': True,\n        \'note\': \'Already missing, cropped or transparent plate pixels cannot be restored by padding.\',\n    }\n    (destination / \'image_preparation.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'source\')\n    parser.add_argument(\'destination\')\n    parser.add_argument(\'--margin\', type=float, default=0.08)\n    parser.add_argument(\'--max-size\', type=int, default=1024)\n    parser.add_argument(\'--reviewed-mask\')\n    arguments = parser.parse_args()\n    print(json.dumps(prepare_full_plate(arguments.source, arguments.destination, arguments.margin,\n                                        arguments.max_size, arguments.reviewed_mask), indent=2))\n', encoding='utf-8')
(SCRATCH_ROOT / 'trellis_native_space_tools.py').write_text('"""HF Space render modes and repeatable GLB export from saved mesh/material data."""\n\nimport argparse\nimport base64\nimport html\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport traceback\n\nfrom PIL import Image, ImageDraw\n\n\nMODES = (\n    (\'normal\', \'Normal\'),\n    (\'clay\', \'Clay render\'),\n    (\'base_color\', \'Base color\'),\n    (\'shaded_forest\', \'HDRI forest\'),\n    (\'shaded_sunset\', \'HDRI sunset\'),\n    (\'shaded_courtyard\', \'HDRI courtyard\'),\n)\n\n\ndef write_gallery(frames, destination):\n    # 1. Keep the six Space render modes usable offline, with the same eight view angles.\n    lengths = {len(frames[key]) for key, _ in MODES}\n    if len(lengths) != 1 or not lengths or min(lengths) < 1:\n        raise ValueError(\'Every Space render mode must have the same nonzero number of views.\')\n    view_count = lengths.pop()\n    images = {\n        key: [\'data:image/png;base64,\' + base64.b64encode(Path(path).read_bytes()).decode()\n              for path in frames[key]]\n        for key, _ in MODES\n    }\n    buttons = \'\'.join(\n        f\'<button type="button" data-mode="{key}" aria-pressed="false">{html.escape(label)}</button>\'\n        for key, label in MODES\n    )\n    document = \'\'\'<!doctype html><html lang="en"><head><meta charset="utf-8">\n    <meta name="viewport" content="width=device-width,initial-scale=1">\n    <title>Full plate — TRELLIS.2 preview modes</title>\n    <style>\n    body{font:16px system-ui,sans-serif;background:#161b24;color:#f4f6fa;margin:0;padding:20px}\n    main{max-width:1000px;margin:auto}h1{font-size:22px}.modes{display:flex;gap:8px;flex-wrap:wrap}\n    button{padding:10px 14px;border:1px solid #737c8d;border-radius:8px;background:#273143;color:inherit;cursor:pointer}\n    button[aria-pressed=true]{background:#245c9b;border-color:#9fcaff}\n    img{display:block;width:100%;height:min(65vh,600px);object-fit:contain;background:#000;margin:16px 0}\n    input{width:min(100%,600px)}p{color:#c7ceda;line-height:1.5}\n    </style></head><body><main><h1>Full plate and food</h1>\n    <div class="modes">__BUTTONS__</div><img id="preview" alt="Generated full plate">\n    <label for="angle">View angle: <span id="view-number"></span> / __COUNT__</label><br>\n    <input id="angle" type="range" min="0" max="__MAX__" value="__DEFAULT__" step="1">\n    <p>All six modes show the same generated object. Normal shows surface direction; clay shows\n    shape; base color shows its color; the HDRI modes show three lighting environments.</p>\n    <script>\n    const frames=__IMAGES__;\n    const angle=document.getElementById(\'angle\');\n    let mode=\'shaded_forest\';\n    function refresh(){\n        document.getElementById(\'preview\').src=frames[mode][Number(angle.value)];\n        document.getElementById(\'view-number\').textContent=Number(angle.value)+1;\n        document.querySelectorAll(\'[data-mode]\').forEach(button=>{\n            button.setAttribute(\'aria-pressed\',String(button.dataset.mode===mode));\n        });\n    }\n    document.querySelectorAll(\'[data-mode]\').forEach(button=>button.addEventListener(\'click\',()=>{\n        mode=button.dataset.mode;refresh();\n    }));\n    angle.addEventListener(\'input\',refresh);refresh();\n    </script></main></body></html>\'\'\'\n    document = document.replace(\'__BUTTONS__\', buttons).replace(\'__COUNT__\', str(view_count))\n    document = document.replace(\'__MAX__\', str(view_count - 1)).replace(\'__DEFAULT__\', str(min(3, view_count - 1)))\n    document = document.replace(\'__IMAGES__\', json.dumps(images))\n    destination = Path(destination)\n    destination.write_text(document, encoding=\'utf-8\')\n    return destination\n\n\ndef load_saved_mesh(output, torch, device=None):\n    import numpy as np\n    from trellis2.representations import MeshWithVoxel\n\n    if device is None:\n        device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    elif isinstance(device, str):\n        device = torch.device(device)\n\n    metadata = json.loads((output / \'raw_mesh_metadata.json\').read_text(encoding=\'utf-8\'))\n    layout = {key: slice(*value) for key, value in metadata[\'attribute_layout\'].items()}\n    with np.load(output / \'raw_mesh_and_materials.npz\', allow_pickle=False) as saved:\n        if \'voxel_shape\' in saved:\n            voxel_shape = torch.Size(saved[\'voxel_shape\'].tolist())\n        else:\n            voxel_shape = torch.Size(metadata[\'voxel_shape\'])\n        mesh = MeshWithVoxel(\n            vertices=torch.from_numpy(saved[\'vertices\']).to(device),\n            faces=torch.from_numpy(saved[\'faces\']).to(device),\n            origin=saved[\'origin\'].tolist(),\n            voxel_size=float(saved[\'voxel_size\'].item() if hasattr(saved[\'voxel_size\'], \'item\') else saved[\'voxel_size\']),\n            coords=torch.from_numpy(saved[\'coords\']).to(device),\n            attrs=torch.from_numpy(saved[\'attributes\']).to(device),\n            voxel_shape=voxel_shape, layout=layout,\n        )\n    if not len(mesh.vertices) or not len(mesh.faces):\n        raise ValueError(\'Saved mesh is empty.\')\n    if not torch.isfinite(mesh.vertices).all() or not torch.isfinite(mesh.attrs).all():\n        raise ValueError(\'Saved mesh/materials contain NaN or Inf.\')\n    return mesh\n\n\ndef render_modes(mesh, configuration, directory, torch):\n    import cv2\n    import numpy as np\n    from trellis2.renderers import EnvMap\n    from trellis2.utils import render_utils\n\n    resolution = int(configuration[\'preview_resolution\'])\n    supersampling = int(configuration[\'preview_ssaa\'])\n    if resolution not in (256, 512, 1024) or supersampling not in (1, 2):\n        raise ValueError(\'Preview resolution must be 256, 512 or 1024; SSAA must be 1 or 2.\')\n    # 2. Use the same official HDRIs, PBR renderer and camera positions as the HF Space.\n    environment_maps = {}\n    for name in (\'forest\', \'sunset\', \'courtyard\'):\n        path = Path(configuration[\'source_dir\']) / \'assets\' / \'hdri\' / (name + \'.exr\')\n        image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)\n        if image is None:\n            raise RuntimeError(f\'The official lighting asset could not be read: {path}\')\n        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n        environment_maps[name] = EnvMap(torch.tensor(image, dtype=torch.float32, device=\'cuda\'))\n    if len(mesh.faces) > 16777216:\n        mesh.simplify(16777216)  # The same nvdiffrast face limit as the Space; saved geometry is unchanged.\n    yaws = (np.linspace(0, 2 * np.pi, 8, endpoint=False) - 16 / 180 * np.pi).tolist()\n    pitches = [20 / 180 * np.pi] * 8\n    extrinsics, intrinsics = render_utils.yaw_pitch_r_fov_to_extrinsics_intrinsics(yaws, pitches, 2, 36)\n    renderer = render_utils.get_renderer(mesh, resolution=resolution, ssaa=supersampling, peel_layers=8)\n    frames = {key: [] for key, _ in MODES}\n    for view, (extrinsic, intrinsic) in enumerate(zip(extrinsics, intrinsics)):\n        print(f\'Rendering all six modes: angle {view + 1}/8\', flush=True)\n        torch.manual_seed(42 + view)\n        result = renderer.render(mesh, extrinsic, intrinsic, envmap=environment_maps)\n        for key, _ in MODES:\n            image = result[key]\n            if not torch.isfinite(image).all():\n                raise RuntimeError(f\'The {key} preview contains NaN/Inf.\')\n            if image.dim() == 2:\n                image = image[None].repeat(3, 1, 1)\n            pixels = np.clip(image.detach().cpu().numpy().transpose(1, 2, 0) * 255, 0, 255).astype(np.uint8)\n            target = directory / f\'{key}_{view:02d}.png\'\n            Image.fromarray(pixels).save(target)\n            frames[key].append(target)\n        del result\n    gallery = write_gallery(frames, directory / \'all_hf_modes.html\')\n    # Keep a compact comparison in notebook output instead of embedding all 48 full images.\n    sheet = Image.new(\'RGB\', (3 * 320, 2 * 350), \'#161b24\')\n    draw = ImageDraw.Draw(sheet)\n    for index, (key, label) in enumerate(MODES):\n        x, y = (index % 3) * 320, (index // 3) * 350\n        with Image.open(frames[key][3]) as image:\n            sheet.paste(image.resize((320, 320), Image.Resampling.LANCZOS), (x, y + 30))\n        draw.text((x + 10, y + 8), label, fill=\'white\')\n    sheet.save(directory / \'all_modes_comparison.jpg\', quality=90)\n    return {\'gallery\': str(gallery), \'modes\': [key for key, _ in MODES], \'views_per_mode\': 8,\n            \'preview_resolution\': resolution, \'preview_ssaa\': supersampling,\n            \'rendered_from\': \'saved raw mesh and material voxels\', \'geometry_changed\': False}\n\n\ndef export_variant(mesh, configuration, directory):\n    faces = int(configuration[\'decimation_target\'])\n    texture = int(configuration[\'texture_size\'])\n    if not 100000 <= faces <= 500000 or faces % 10000:\n        raise ValueError(\'Use the Space face-count range: 100000–500000, in steps of 10000.\')\n    if texture not in (1024, 2048, 3072, 4096):\n        raise ValueError(\'Use a Space texture size: 1024, 2048, 3072 or 4096.\')\n    import o_voxel\n    # 3. Re-export from unmodified saved geometry; never chain variants or regenerate the food.\n    asset = o_voxel.postprocess.to_glb(\n        vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,\n        coords=mesh.coords, attr_layout=mesh.layout, voxel_size=mesh.voxel_size,\n        aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],\n        decimation_target=faces, texture_size=texture,\n        remesh=bool(configuration[\'remesh\']), remesh_band=1, remesh_project=0,\n        use_tqdm=True,\n    )\n    target = directory / f\'full_plate_faces{faces}_texture{texture}_remesh{int(configuration["remesh"])}.glb\'\n    asset.export(str(target), extension_webp=True)\n    with target.open(\'rb\') as stream:\n        if stream.read(4) != b\'glTF\' or target.stat().st_size <= 20:\n            raise RuntimeError(\'Variant export did not create a valid GLB container.\')\n    return {\'output\': str(target), \'decimation_target\': faces, \'texture_size\': texture,\n            \'remesh\': bool(configuration[\'remesh\'])}\n\n\ndef main(config_path, action):\n    configuration = json.loads(Path(config_path).read_text(encoding=\'utf-8\'))\n    output = Path(configuration[\'output_dir\'])\n    directory = Path(configuration[\'result_dir\'])\n    directory.mkdir(parents=True, exist_ok=True)\n    status = {\'status\': \'running\', \'action\': action, \'started_at\': time.time()}\n    status_path = directory / \'status.json\'\n    status_path.write_text(json.dumps(status, indent=2), encoding=\'utf-8\')\n    try:\n        os.environ[\'OPENCV_IO_ENABLE_OPENEXR\'] = \'1\'\n        os.environ[\'ATTN_BACKEND\'] = \'sdpa\'\n        os.environ[\'SPARSE_CONV_BACKEND\'] = \'none\'\n        os.environ[\'SPARSE_ATTN_BACKEND\'] = \'xformers\'\n        os.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\n        sys.path.insert(0, configuration[\'source_dir\'])\n        import torch\n        from trellis_native_compat import install_adapters\n        install_adapters()\n        # PBR renderer imports this function inside render(); patch its original module too.\n        sampling = importlib.import_module(\'flex_gemm.ops.grid_sample\')\n        from trellis_native_compat import sparse_grid_sample\n        sampling.grid_sample_3d = sparse_grid_sample\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Rendering and native GLB export require the Kaggle CUDA session.\')\n        torch.cuda.set_device(0)\n        device = torch.device(\'cuda:0\')\n        with torch.inference_mode():\n            mesh = load_saved_mesh(output, torch, device=device)\n            if action == \'render\':\n                result = render_modes(mesh, configuration, directory, torch)\n            else:\n                result = export_variant(mesh, configuration, directory)\n        status.update(result)\n        status.update(status=\'success\', ended_at=time.time())\n        print(json.dumps(status, indent=2), flush=True)\n    except Exception as error:\n        status.update(status=\'failed\', message=str(error), error_type=type(error).__name__, ended_at=time.time())\n        traceback.print_exc()\n        raise\n    finally:\n        status_path.write_text(json.dumps(status, indent=2), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'action\', choices=(\'render\', \'export\'))\n    parser.add_argument(\'config\')\n    arguments = parser.parse_args()\n    main(arguments.config, arguments.action)\n', encoding='utf-8')
(SCRATCH_ROOT / 'test_trellis_native_compat.py').write_text('"""Small numerical checks; run on CPU locally and CUDA inside the notebook."""\n\nimport itertools\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport unittest\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\n\nimport numpy as np\nfrom PIL import Image\n\ntest_packages = Path(__file__).with_name(\'trellis-native-test-packages\')\nif test_packages.exists():\n    sys.path.insert(0, str(test_packages))\n\nimport torch\nimport torch.nn.functional as F\nfrom trellis_native_compat import CoordinateLookup, submanifold_convolution, sparse_attention, sparse_grid_sample\nfrom trellis_prepare_full_plate import prepare_full_plate\nfrom trellis_native_space_tools import load_saved_mesh, write_gallery, export_variant, MODES\n\n\nclass MeshWithVoxel:\n    """Test double for metadata I/O; native rendering still requires the official class."""\n    def __init__(self, **values):\n        self.__dict__.update(values)\n        self.origin = torch.tensor(values[\'origin\'], dtype=torch.float32)\n\n\nclass Sequence:\n    def __init__(self, feats, lengths):\n        self.feats = feats\n        self.layout = []\n        start = 0\n        for length in lengths:\n            self.layout.append(slice(start, start + length))\n            start += length\n\n    def replace(self, feats):\n        return Sequence(feats, [s.stop - s.start for s in self.layout])\n\n\nclass NativeAdaptersTest(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(42)\n        self.device = os.environ.get(\'TRELLIS_TEST_DEVICE\', \'cpu\')\n\n    def test_coordinate_boundaries_do_not_alias_another_batch(self):\n        coords = torch.tensor([[0, 0, 0, 0], [0, 1, 2, 2], [1, 0, 0, 0]], device=self.device)\n        lookup = CoordinateLookup(coords, (2, 3, 3))\n        query = torch.tensor([[0, 2, 0, 0], [0, -1, 0, 0], [1, 0, 0, 0], [0, 1, 2, 2]], device=self.device)\n        self.assertEqual(lookup.find(query).tolist(), [-1, -1, 2, 1])\n\n    def test_sparse_convolution_matches_dense_with_holes_and_dilation(self):\n        coords = torch.tensor(list(itertools.product(range(2), range(4), range(3), range(5))), device=self.device)\n        coords = coords[torch.randperm(len(coords), device=self.device)[:67]]\n        features = torch.randn(len(coords), 3, device=self.device)\n        weight = torch.randn(4, 3, 3, 3, 3, device=self.device)\n        bias = torch.randn(4, device=self.device)\n        dense = torch.zeros(2, 3, 4, 3, 5, device=self.device)\n        dense[coords[:, 0], :, coords[:, 1], coords[:, 2], coords[:, 3]] = features\n        for dilation in (1, 2):\n            expected = F.conv3d(dense, weight, bias, padding=dilation, dilation=dilation)\n            expected = expected[coords[:, 0], :, coords[:, 1], coords[:, 2], coords[:, 3]]\n            actual, cache = submanifold_convolution(\n                features, coords, weight.permute(0, 2, 3, 4, 1).contiguous(), bias,\n                (dilation,) * 3, chunk_size=7,\n            )\n            torch.testing.assert_close(actual, expected, rtol=2e-5, atol=2e-5)\n            cached, _ = submanifold_convolution(\n                features, coords, weight.permute(0, 2, 3, 4, 1).contiguous(), bias,\n                (dilation,) * 3, neighbor_map=cache, chunk_size=11,\n            )\n            torch.testing.assert_close(actual, cached)\n\n    def test_attention_varlen_self_and_cross_match_independent_sequences(self):\n        query = torch.randn(8, 2, 4, device=self.device)\n        key = torch.randn(10, 2, 4, device=self.device)\n        value = torch.randn(10, 2, 6, device=self.device)\n        q = Sequence(query, [3, 5])\n        k = Sequence(key, [6, 4])\n        v = Sequence(value, [6, 4])\n        actual = sparse_attention(q=q, k=k, v=v)\n        for qs, ks in zip(q.layout, k.layout):\n            expected = F.scaled_dot_product_attention(\n                query[qs].transpose(0, 1)[None], key[ks].transpose(0, 1)[None], value[ks].transpose(0, 1)[None]\n            )[0].transpose(0, 1)\n            torch.testing.assert_close(actual.feats[qs], expected)\n        qkv = Sequence(torch.randn(8, 3, 2, 4, device=self.device), [3, 5])\n        self.assertEqual(sparse_attention(qkv).feats.shape, (8, 2, 4))\n\n    def test_attention_mixed_dense_sparse_and_query_chunking(self):\n        q = torch.randn(2, 270, 2, 4, device=self.device)\n        kv = Sequence(torch.randn(13, 2, 2, 4, device=self.device), [6, 7])\n        actual = sparse_attention(q, kv)\n        for index, key_slice in enumerate(kv.layout):\n            key, value = kv.feats[key_slice].unbind(1)\n            expected = F.scaled_dot_product_attention(\n                q[index].transpose(0, 1)[None], key.transpose(0, 1)[None], value.transpose(0, 1)[None]\n            )[0].transpose(0, 1)\n            torch.testing.assert_close(actual[index], expected)\n        sparse_q = Sequence(torch.randn(9, 2, 4, device=self.device), [4, 5])\n        dense_kv = torch.randn(2, 7, 2, 2, 4, device=self.device)\n        actual = sparse_attention(sparse_q, dense_kv)\n        for index, qs in enumerate(sparse_q.layout):\n            key, value = dense_kv[index].unbind(1)\n            expected = F.scaled_dot_product_attention(\n                sparse_q.feats[qs].transpose(0, 1)[None], key.transpose(0, 1)[None], value.transpose(0, 1)[None]\n            )[0].transpose(0, 1)\n            torch.testing.assert_close(actual.feats[qs], expected)\n\n    def test_voxel_centers_interpolation_missing_sites_and_outside(self):\n        coords = torch.tensor(list(itertools.product(range(2), range(2), range(2), range(2))), device=self.device)\n        features = (coords[:, 1:].float().sum(1) + 10 * coords[:, 0]).unsqueeze(1)\n        queries = torch.tensor([[[0.5, 0.5, 0.5], [1, 1, 1], [5, 5, 5]],\n                                [[0.5, 0.5, 0.5], [1, 1, 1], [5, 5, 5]]], device=self.device)\n        output = sparse_grid_sample(features, coords, (2, 1, 2, 2, 2), queries, chunk_size=2)\n        expected = torch.tensor([[[0], [1.5], [0]], [[10], [11.5], [0]]], dtype=torch.float32, device=self.device)\n        torch.testing.assert_close(output, expected)\n        output = sparse_grid_sample(features[:1] + 7, coords[:1], (1, 1, 2, 2, 2), queries[:1])\n        torch.testing.assert_close(output[0, :2], torch.full((2, 1), 7.0, device=self.device))\n\n    def test_nearest_sampling_truncates_negative_fractional_queries(self):\n        coords = torch.tensor([[0, 0, 0, 0]], device=self.device)\n        features = torch.tensor([[3.5, -1.25]], device=self.device)\n        queries = torch.tensor([[[-0.25, 0.25, 0.25], [-1.25, 0.25, 0.25], [0.75, 0.25, 0.25]]], device=self.device)\n        output = sparse_grid_sample(features, coords, (1, 2, 2, 2, 2), queries, mode=\'nearest\')\n        expected = torch.tensor([[[3.5, -1.25], [0.0, 0.0], [3.5, -1.25]]], device=self.device)\n        torch.testing.assert_close(output, expected)\n\n\nclass FullPlatePreparationTest(unittest.TestCase):\n    def setUp(self):\n        self.temp_dir = tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent)\n        self.dest = Path(self.temp_dir.name)\n\n    def tearDown(self):\n        self.temp_dir.cleanup()\n\n    def test_full_plate_padding_and_preservation(self):\n        source_path = self.dest / \'test_plate.png\'\n        image = Image.new(\'RGB\', (200, 150), color=(180, 140, 100))\n        image.save(source_path)\n\n        report = prepare_full_plate(source_path, self.dest, margin=0.1, max_size=512)\n        self.assertFalse(report[\'foreground_cropped\'])\n        self.assertFalse(report[\'automatic_background_removal\'])\n        self.assertTrue(report[\'existing_alpha_preserved\'])\n        self.assertEqual(report[\'original_size\'], [200, 150])\n        self.assertEqual(report[\'prepared_size\'], [240, 240])\n        self.assertTrue((self.dest / \'prepared_image.png\').is_file())\n        self.assertTrue((self.dest / \'full_plate_rgba.png\').is_file())\n        self.assertTrue((self.dest / \'image_preparation.json\').is_file())\n\n    def test_transparent_input_rejected(self):\n        source_path = self.dest / \'transparent.png\'\n        Image.new(\'RGBA\', (64, 64), (0, 0, 0, 0)).save(source_path)\n        with self.assertRaises(ValueError):\n            prepare_full_plate(source_path, self.dest)\n\n    def test_mask_dimension_mismatch(self):\n        source_path = self.dest / \'source.png\'\n        mask_path = self.dest / \'mask.png\'\n        Image.new(\'RGB\', (100, 100), (255, 255, 255)).save(source_path)\n        Image.new(\'L\', (50, 50), 255).save(mask_path)\n        with self.assertRaises(ValueError):\n            prepare_full_plate(source_path, self.dest, reviewed_mask=mask_path)\n\n\nclass SpaceToolsMeshContractTest(unittest.TestCase):\n    def setUp(self):\n        self.temp_dir = tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent)\n        self.dir = Path(self.temp_dir.name)\n\n    def tearDown(self):\n        self.temp_dir.cleanup()\n\n    def test_mesh_save_load_roundtrip(self):\n        vertices = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=torch.float32)\n        faces = torch.tensor([[0, 1, 2]], dtype=torch.int32)\n        coords = torch.tensor([[0, 0, 0, 0], [0, 1, 1, 1]], dtype=torch.int32)\n        attrs = torch.tensor([[0.5, 0.2, 0.1, 0.8], [0.1, 0.9, 0.3, 0.4]], dtype=torch.float32)\n        origin = [-0.5, -0.5, -0.5]\n        voxel_size = 0.01\n        voxel_shape = torch.Size([64, 64, 64])\n        layout = {\'color\': slice(0, 3), \'roughness\': slice(3, 4)}\n\n        np.savez_compressed(\n            self.dir / \'raw_mesh_and_materials.npz\',\n            vertices=vertices.numpy(), faces=faces.numpy(),\n            coords=coords.numpy(), attributes=attrs.numpy(),\n            voxel_size=np.array(voxel_size), origin=np.array(origin),\n            voxel_shape=np.array(list(voxel_shape)),\n        )\n        layout_dict = {k: [v.start if v.start is not None else 0, v.stop] for k, v in layout.items()}\n        metadata = {\n            \'attribute_layout\': layout_dict,\n            \'voxel_size\': voxel_size,\n            \'voxel_shape\': list(voxel_shape),\n            \'origin\': origin,\n        }\n        (self.dir / \'raw_mesh_metadata.json\').write_text(json.dumps(metadata, indent=2), encoding=\'utf-8\')\n\n        with patch.dict(sys.modules, {\'trellis2.representations\': SimpleNamespace(MeshWithVoxel=MeshWithVoxel)}):\n            loaded_mesh = load_saved_mesh(self.dir, torch, device=torch.device(\'cpu\'))\n        torch.testing.assert_close(loaded_mesh.vertices, vertices)\n        torch.testing.assert_close(loaded_mesh.faces, faces)\n        torch.testing.assert_close(loaded_mesh.coords, coords)\n        torch.testing.assert_close(loaded_mesh.attrs, attrs)\n        self.assertEqual(loaded_mesh.origin.tolist(), origin)\n        self.assertEqual(loaded_mesh.voxel_size, voxel_size)\n        self.assertEqual(loaded_mesh.voxel_shape, voxel_shape)\n        self.assertEqual(loaded_mesh.layout, layout)\n\n    def test_write_gallery_html(self):\n        frames = {}\n        for key, _ in MODES:\n            frames[key] = []\n            for i in range(8):\n                p = self.dir / f\'{key}_{i:02d}.png\'\n                Image.new(\'RGB\', (16, 16), color=(i * 20, 100, 150)).save(p)\n                frames[key].append(p)\n        gallery_path = self.dir / \'all_hf_modes.html\'\n        result = write_gallery(frames, gallery_path)\n        self.assertTrue(result.is_file())\n        content = result.read_text(encoding=\'utf-8\')\n        for key, label in MODES:\n            self.assertIn(label, content)\n            self.assertIn(f\'data-mode="{key}"\', content)\n        for placeholder in (\'__MAX__\', \'__IMAGES__\', \'__BUTTONS__\', \'__COUNT__\', \'__DEFAULT__\'):\n            self.assertNotIn(placeholder, content)\n\n    def test_variant_parameter_validation(self):\n        mesh = MeshWithVoxel(\n            vertices=torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=torch.float32),\n            faces=torch.tensor([[0, 1, 2]], dtype=torch.int32),\n            origin=[0.0, 0.0, 0.0], voxel_size=0.01,\n            coords=torch.zeros((1, 4), dtype=torch.int32),\n            attrs=torch.zeros((1, 4), dtype=torch.float32),\n            voxel_shape=torch.Size([32, 32, 32]),\n        )\n        invalid_configs = [\n            {\'decimation_target\': 90000, \'texture_size\': 1024, \'remesh\': False},\n            {\'decimation_target\': 510000, \'texture_size\': 1024, \'remesh\': False},\n            {\'decimation_target\': 105000, \'texture_size\': 1024, \'remesh\': False},\n            {\'decimation_target\': 100000, \'texture_size\': 512, \'remesh\': False},\n            {\'decimation_target\': 100000, \'texture_size\': 2000, \'remesh\': False},\n        ]\n        for cfg in invalid_configs:\n            with self.subTest(cfg=cfg):\n                with self.assertRaises(ValueError):\n                    export_variant(mesh, cfg, self.dir)\n\n\nif __name__ == \'__main__\':\n    unittest.main(verbosity=2)\n', encoding='utf-8')
(SCRATCH_ROOT / 'test_trellis_full_plate.py').write_text('"""CPU checks for complete-plate input preservation and the six-mode gallery."""\n\nimport base64\nimport json\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unittest\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nfrom trellis_prepare_full_plate import prepare_full_plate\nfrom trellis_native_space_tools import MODES, write_gallery\n\n\nclass FullPlateTest(unittest.TestCase):\n    def setUp(self):\n        self.temporary = tempfile.TemporaryDirectory(dir=Path(__file__).resolve().parent)\n        self.addCleanup(self.temporary.cleanup)\n        self.root = Path(self.temporary.name)\n\n    def test_opaque_image_retains_every_pixel_including_rim_at_all_four_edges(self):\n        image = Image.new(\'RGB\', (96, 64), \'#534132\')\n        draw = ImageDraw.Draw(image)\n        draw.ellipse((0, 0, 95, 63), fill=\'white\')\n        draw.ellipse((30, 20, 65, 43), fill=\'green\')\n        source = self.root / \'plate.png\'\n        image.save(source)\n        output = self.root / \'prepared\'\n        report = prepare_full_plate(source, output)\n        left, top = report[\'image_offset\']\n        with Image.open(output / \'prepared_image.png\') as prepared:\n            # Every original pixel survives, including background and the complete white rim.\n            np.testing.assert_array_equal(np.array(prepared)[top:top + 64, left:left + 96], np.array(image))\n            self.assertGreater(left, 0)\n            self.assertGreater(top, 0)\n        self.assertFalse(report[\'automatic_background_removal\'])\n        self.assertFalse(report[\'foreground_cropped\'])\n\n    def test_translucent_plate_rim_is_not_discarded_by_an_alpha_threshold(self):\n        image = Image.new(\'RGBA\', (70, 50), (220, 220, 255, 40))\n        ImageDraw.Draw(image).ellipse((20, 12, 49, 37), fill=(80, 150, 30, 255))\n        source = self.root / \'transparent_plate.png\'\n        image.save(source)\n        output = self.root / \'prepared\'\n        report = prepare_full_plate(source, output)\n        left, top = report[\'image_offset\']\n        with Image.open(output / \'full_plate_rgba.png\') as prepared:\n            np.testing.assert_array_equal(np.array(prepared)[top:top + 50, left:left + 70], np.array(image))\n\n    def test_large_image_scales_the_whole_frame_and_retains_all_corner_markers(self):\n        image = Image.new(\'RGB\', (1600, 1000), \'white\')\n        draw = ImageDraw.Draw(image)\n        markers = [((0, 0, 199, 199), \'red\'), ((1400, 0, 1599, 199), \'green\'),\n                   ((0, 800, 199, 999), \'blue\'), ((1400, 800, 1599, 999), \'yellow\')]\n        for box, color in markers:\n            draw.rectangle(box, fill=color)\n        source = self.root / \'large.png\'\n        image.save(source)\n        output = self.root / \'prepared\'\n        report = prepare_full_plate(source, output)\n        width, height = report[\'resized_image_size\']\n        left, top = report[\'image_offset\']\n        self.assertEqual(report[\'prepared_size\'], [1024, 1024])\n        self.assertAlmostEqual(width / height, 1.6, places=2)\n        with Image.open(output / \'prepared_image.png\') as prepared:\n            actual = np.array(prepared)\n            samples = [actual[top + 10, left + 10], actual[top + 10, left + width - 11],\n                       actual[top + height - 11, left + 10], actual[top + height - 11, left + width - 11]]\n        np.testing.assert_array_equal(samples, [[255, 0, 0], [0, 128, 0], [0, 0, 255], [255, 255, 0]])\n\n    def test_empty_and_mismatched_masks_are_rejected(self):\n        source = self.root / \'plate.png\'\n        Image.new(\'RGBA\', (80, 60), \'white\').save(source)\n        mask = self.root / \'mask.png\'\n        Image.new(\'L\', (20, 20), 255).save(mask)\n        with self.assertRaisesRegex(ValueError, \'dimensions\'):\n            prepare_full_plate(source, self.root / \'bad_size\', reviewed_mask=mask)\n        Image.new(\'L\', (80, 60), 0).save(mask)\n        with self.assertRaisesRegex(ValueError, \'fully transparent\'):\n            prepare_full_plate(source, self.root / \'empty\', reviewed_mask=mask)\n\n    def test_gallery_contains_all_six_modes_and_all_eight_views_offline(self):\n        frames = {}\n        for mode_index, (key, _) in enumerate(MODES):\n            frames[key] = []\n            for view in range(8):\n                path = self.root / f\'{key}_{view}.png\'\n                Image.new(\'RGB\', (8, 8), (mode_index * 40, view * 30, 128)).save(path)\n                frames[key].append(path)\n        document = write_gallery(frames, self.root / \'gallery.html\').read_text(encoding=\'utf-8\')\n        images = json.loads(re.search(r\'const frames=(.*);\', document).group(1))\n        self.assertEqual(set(images), {\'normal\', \'clay\', \'base_color\', \'shaded_forest\', \'shaded_sunset\', \'shaded_courtyard\'})\n        self.assertTrue(all(len(views) == 8 for views in images.values()))\n        self.assertEqual(document.count(\'data-mode="\'), 6)\n        self.assertNotIn(\'<script src=\', document)\n        for key, _ in MODES:\n            for view, encoded in enumerate(images[key]):\n                self.assertEqual(base64.b64decode(encoded.split(\',\', 1)[1]), frames[key][view].read_bytes())\n\n    def test_gallery_rejects_incomplete_mode_angles(self):\n        frames = {key: [self.root / \'unused.png\'] for key, _ in MODES}\n        frames[\'clay\'] = []\n        with self.assertRaisesRegex(ValueError, \'same nonzero\'):\n            write_gallery(frames, self.root / \'incomplete.html\')\n\n\nif __name__ == \'__main__\':\n    unittest.main(verbosity=2)\n', encoding='utf-8')
RUNTIME_ENV = BUILD_ENV.copy()
RUNTIME_ENV['PYTHONPATH'] = (
    (str(TRELLIS_SOURCE) + os.pathsep if TRELLIS_SOURCE.exists() else '') + str(SCRATCH_ROOT)
)
RUNTIME_ENV['ATTN_BACKEND'] = 'sdpa'
RUNTIME_ENV['SPARSE_CONV_BACKEND'] = 'none'
RUNTIME_ENV['TRELLIS_TEST_DEVICE'] = 'cuda:0' if TRELLIS_BUILD_OK else 'cpu'
run([PYTHON, SCRATCH_ROOT / 'test_trellis_full_plate.py'], env=RUNTIME_ENV)
run([PYTHON, SCRATCH_ROOT / 'test_trellis_native_compat.py'], env=RUNTIME_ENV)
if TRELLIS_BUILD_OK:
    smoke = 'from trellis_native_compat import install_adapters; install_adapters(); import cumesh, o_voxel, nvdiffrast.torch'
    if NVDIFFREC_AVAILABLE:
        smoke += '; from trellis2.renderers import EnvMap; from nvdiffrec_render.light import EnvironmentLight'
    smoke += '; print("Native TRELLIS imports passed")'
    run([PYTHON, '-c', smoke], env=RUNTIME_ENV)

## 6. Run `TRELLIS.2-4B` (when DINOv3 access is enabled)

If `DINOV3_OK` and `TRELLIS_BUILD_OK` are true, runs Microsoft `TRELLIS.2-4B` at 512 resolution per input
image (reusing the first run's pinned checkpoint revisions on the second input) and records PBR-textured
GLBs (`appearance: "pbr_texture"`). Otherwise records a clean `skipped` status in the manifest.


In [ ]:
SEED = 42
MARGIN = 0.08
MAX_SIZE = 1024
SETTINGS = {
    'seed': SEED,
    'sampling_steps': 12,
    'samplers': {
        'sparse_structure': {'steps': 12, 'guidance_strength': 7.5, 'guidance_rescale': 0.7, 'rescale_t': 5.0},
        'shape': {'steps': 12, 'guidance_strength': 7.5, 'guidance_rescale': 0.5, 'rescale_t': 3.0},
        'material': {'steps': 12, 'guidance_strength': 1.0, 'guidance_rescale': 0.0, 'rescale_t': 3.0},
    },
    'texture_size': 1024,
    'decimation_target': 100000,
    'remesh': False,
    'save_latents': False,
}

import signal as _signal

TRELLIS_WORKERS = {}


def _run_logged_bounded(cmd, env, log_stream, timeout, label):
    process = subprocess.Popen(
        [str(v) for v in cmd],
        env=env,
        stdout=log_stream,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    try:
        return process.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, _signal.SIGKILL)
        except OSError:
            process.kill()
        try:
            process.wait(timeout=10)
        except subprocess.SubprocessError:
            pass
        raise DemoError(f'{label} timed out after {timeout}s.') from None


def trellis_generation(folder, source_image, runtime, checkpoint_revisions):
    folder.mkdir(parents=True, exist_ok=True)
    prepare = [PYTHON, SCRATCH_ROOT / 'trellis_prepare_full_plate.py', source_image,
               folder, '--margin', str(MARGIN), '--max-size', str(MAX_SIZE)]
    with (folder / 'native_generation.log').open('w', encoding='utf-8') as log:
        prep_rc = _run_logged_bounded(prepare, RUNTIME_ENV, log, timeout=120, label='Whole-plate preparation')
        if prep_rc:
            raise DemoError(f'Whole-plate preparation exited with code {prep_rc}.')
        config = {
            **SETTINGS,
            'source_dir': str(TRELLIS_SOURCE),
            'output_dir': str(folder),
            'prepared_image': str(folder / 'prepared_image.png'),
            'model_revision': checkpoint_revisions.get(TRELLIS_MODEL_REPO, TRELLIS_MODEL_REVISION),
            'checkpoint_revisions': dict(checkpoint_revisions),
        }
        config_path = folder / 'generation_config.json'
        config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
        run_rc = _run_logged_bounded(
            [PYTHON, SCRATCH_ROOT / 'trellis_native_runner.py', config_path],
            runtime,
            log,
            timeout=3600,
            label='TRELLIS runner',
        )
    status_path = folder / 'run_status.json'
    status = json.loads(status_path.read_text(encoding='utf-8')) if status_path.is_file() else {}
    status = {k: v for k, v in status.items() if k in (
        'status', 'error_type', 'generation_seconds', 'export_seconds', 'peak_gpu0_gib', 'encoder_gpu'
    )}
    status_path.write_text(json.dumps(status, indent=2), encoding='utf-8')
    if run_rc or status.get('status') != 'success':
        raise DemoError(f"TRELLIS exited with code {run_rc} ({status.get('error_type', 'unknown error')}).")
    validate_glb(folder / 'generated_original.glb')
    return status


checkpoint_revisions = {
    DINOV3_REPO: DINOV3_REVISION,
    TRELLIS_MODEL_REPO: TRELLIS_MODEL_REVISION,
}
for label in ACTIVE_LABELS:
    in_entry = image_results[label]
    key = f'{label}/trellis2-4b'
    rec = {
        'input_label': label,
        'image_source': in_entry['image_source'],
        'image_model': in_entry['model'],
        'backend': 'TRELLIS.2-4B',
        'appearance': 'pbr_texture',
        'artifact_paths': [],
    }
    if not (DINOV3_OK and TRELLIS_BUILD_OK):
        skip_reason = TRELLIS_SKIP_REASON if (DINOV3_OK and not TRELLIS_BUILD_OK and globals().get('TRELLIS_SKIP_REASON')) else DINOV3_NOTE
        rec.update(status='skipped', error=f'Skipped ({skip_reason})')
        RUNS_3D[key] = rec
        continue
    work_folder = SCRATCH_ROOT / 'trellis_runs' / label
    runtime = RUNTIME_ENV.copy()
    if HF_TOKEN:
        runtime['HF_TOKEN'] = HF_TOKEN
    try:
        stats = trellis_generation(work_folder, in_entry['image_path'], runtime, checkpoint_revisions)
        run_revs = json.loads((work_folder / 'model_revision.json').read_text(encoding='utf-8')).get('revisions', {})
        checkpoint_revisions.update(run_revs)
        REVISIONS['trellis_checkpoints'] = dict(checkpoint_revisions)
        out_glb = RESULTS_DIR / label / 'trellis2_4b.glb'
        shutil.copyfile(work_folder / 'generated_original.glb', out_glb)
        glb_meta = validate_glb(out_glb)
        rec.update(
            status='success',
            appearance='pbr_texture' if glb_meta['has_texture'] else 'untextured_mesh',
            checkpoint_revisions=dict(checkpoint_revisions),
            metrics={**stats, **glb_meta},
            artifact_paths=[out_glb],
        )
        TRELLIS_WORKERS[label] = work_folder
        display(FileLink(str(out_glb)))
    except Exception as exc:
        rec.update(status='failed', error=str(exc))
        print(key, 'failed:', exc)
    RUNS_3D[key] = rec

## 7. Optional `TRELLIS.2` HF Space 6-mode render (only when `DINOV3_OK` & `NVDIFFREC_AVAILABLE`)

Renders the 6 official TRELLIS.2 PBR preview filters when both DINOv3 access and `nvdiffrec_render` succeeded.


In [ ]:
if DINOV3_OK and TRELLIS_BUILD_OK and NVDIFFREC_AVAILABLE and TRELLIS_WORKERS:
    for label, work_folder in TRELLIS_WORKERS.items():
        render_dir = RESULTS_DIR / label / 'trellis_hf_modes'
        space_cfg = work_folder / 'space_render_config.json'
        space_cfg.write_text(json.dumps({
            'source_dir': str(TRELLIS_SOURCE),
            'output_dir': str(work_folder),
            'result_dir': str(render_dir),
            'preview_resolution': 512,
            'preview_ssaa': 1,
        }, indent=2), encoding='utf-8')
        try:
            render_env = RUNTIME_ENV.copy()
            render_env['CUDA_VISIBLE_DEVICES'] = '1' if GPU_COUNT > 1 else '0'
            run([PYTHON, SCRATCH_ROOT / 'trellis_native_space_tools.py', 'render', space_cfg], env=render_env)
            montage = render_dir / 'all_modes_comparison.jpg'
            if montage.is_file():
                display(DisplayImage(filename=str(montage)))
        except Exception as err:
            print('Optional TRELLIS Space render skipped:', type(err).__name__)
else:
    print('TRELLIS HF 6-mode render skipped (requires DINOV3_OK, NVDIFFREC_AVAILABLE, and a TRELLIS GLB).')


## 8. HF TRELLIS Studio Inspection (`Normal` / `Clay` / `Base Color` / `HDRI`), Viewer & Docker Blueprint

Uses pure-Python software rendering (`trimesh` + `pymeshlab` + `PIL`) to produce both 3-view comparison strips
(`comparison_sheet.png`) and **HF TRELLIS Demo-style 4-mode studio renders** (`hf_modes_sheet.png`), builds an
interactive studio viewer (`comparison_viewer.html`) with raw-vs-mobile `<=35k` GLB toggles, writes per-dish
`worker_output.json` receipts, and generates the reproducible **`docker_blueprint/`** (`Dockerfile.worker.template`,
`worker_entrypoint.py`, `requirements.lock.txt`, `pinned_revisions.json`) packaged into `dish-3d-comparison.zip`.


In [ ]:
import html as html_lib
import math
from PIL import ImageDraw
import pymeshlab
import trimesh


def make_checkerboard(size, tile=16, c1=(28, 34, 44), c2=(42, 49, 62)):
    w, h = size
    ys, xs = np.indices((h, w))
    pat = ((xs // tile) + (ys // tile)) % 2 == 0
    board = np.empty((h, w, 3), dtype=np.uint8)
    board[pat] = c1
    board[~pat] = c2
    return Image.fromarray(board, mode='RGB')


def _load_decimated_mesh_data(glb_path, target_faces=10000):
    scene = trimesh.load(str(glb_path), force='scene', process=False)
    meshes = [g for g in scene.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    if not meshes:
        raise ValueError(f'No triangle mesh found in {glb_path}')
    combined = trimesh.util.concatenate(meshes) if len(meshes) > 1 else meshes[0].copy()
    verts = np.asarray(combined.vertices, dtype=np.float32)
    faces = np.asarray(combined.faces, dtype=np.int32)
    has_vc = combined.visual.kind == 'vertex' and hasattr(combined.visual, 'vertex_colors')
    face_rgb = None
    if has_vc:
        v_col = np.asarray(combined.visual.vertex_colors, dtype=np.float32)[:, :3]
        v_srgb = np.power(np.clip(v_col / 255.0, 0.0, 1.0), 1.0 / 2.2) * 255.0
        face_rgb = v_srgb[faces].mean(axis=1)

    if len(faces) > target_faces:
        try:
            ms = pymeshlab.MeshSet()
            if has_vc:
                vc = np.asarray(combined.visual.vertex_colors, dtype=np.float64) / 255.0
                if vc.shape[1] == 3:
                    vc = np.concatenate([vc, np.ones((len(vc), 1))], axis=1)
                pm = pymeshlab.Mesh(
                    vertex_matrix=verts.astype(np.float64),
                    face_matrix=faces.astype(np.int32),
                    v_color_matrix=vc,
                )
            else:
                pm = pymeshlab.Mesh(
                    vertex_matrix=verts.astype(np.float64),
                    face_matrix=faces.astype(np.int32),
                )
            ms.add_mesh(pm)
            ms.apply_filter('meshing_decimation_quadric_edge_collapse', targetfacenum=target_faces, preserveboundary=True)
            cm = ms.current_mesh()
            verts = cm.vertex_matrix().astype(np.float32)
            faces = cm.face_matrix().astype(np.int32)
            if has_vc and cm.has_vertex_color():
                v_lin = np.clip(cm.vertex_color_matrix()[:, :3], 0.0, 1.0).astype(np.float32)
                v_srgb = np.power(v_lin, 1.0 / 2.2) * 255.0
                face_rgb = v_srgb[faces].mean(axis=1)
            else:
                face_rgb = None
        except Exception:
            if len(faces) > 40000:
                step = max(1, len(faces) // 40000)
                faces = faces[::step]
                if face_rgb is not None:
                    face_rgb = face_rgb[::step]

    if face_rgb is None:
        face_rgb = np.full((len(faces), 3), [214.0, 200.0, 184.0], dtype=np.float32)

    vmin, vmax = verts.min(axis=0), verts.max(axis=0)
    center = 0.5 * (vmin + vmax)
    scale = max(float(np.linalg.norm(vmax - vmin)) * 0.5, 1e-5)
    verts = (verts - center) / scale
    return verts, faces, face_rgb


def _project_triangles(verts, faces, face_rgb, yaw_deg, pitch_deg):
    yaw, pitch = math.radians(yaw_deg), math.radians(pitch_deg)
    ry = np.array([[math.cos(yaw), 0, math.sin(yaw)], [0, 1, 0], [-math.sin(yaw), 0, math.cos(yaw)]], dtype=np.float32)
    rx = np.array([[1, 0, 0], [0, math.cos(pitch), -math.sin(pitch)], [0, math.sin(pitch), math.cos(pitch)]], dtype=np.float32)
    rv = (verts @ ry.T) @ rx.T
    tri = rv[faces]
    e1 = tri[:, 1] - tri[:, 0]
    e2 = tri[:, 2] - tri[:, 0]
    norms = np.cross(e1, e2)
    nlen = np.linalg.norm(norms, axis=1, keepdims=True)
    valid = nlen[:, 0] > 1e-10
    tri, frgb, norms = tri[valid], face_rgb[valid], norms[valid] / nlen[valid]
    z_order = np.argsort(tri[:, :, 2].mean(axis=1))
    return tri[z_order], frgb[z_order], norms[z_order]


def render_glb_three_views(glb_path, out_png, tile_size=260):
    verts, faces, face_rgb = _load_decimated_mesh_data(glb_path)
    angles = [('Front 3/4', 35, 28), ('Side +90', 125, 22), ('Top 3/4', 35, 68)]
    strip = Image.new('RGB', (tile_size * 3, tile_size + 26), (26, 30, 38))
    s_draw = ImageDraw.Draw(strip)
    key_light = np.array([0.35, 0.78, 0.52], dtype=np.float32)
    key_light /= np.linalg.norm(key_light)
    fill_light = np.array([-0.55, 0.35, 0.75], dtype=np.float32)
    fill_light /= np.linalg.norm(fill_light)

    for idx, (title, yaw_deg, pitch_deg) in enumerate(angles):
        tri, frgb, norms = _project_triangles(verts, faces, face_rgb, yaw_deg, pitch_deg)
        d_key = np.clip((norms * key_light).sum(axis=1), 0.0, 1.0)
        d_fill = np.clip((norms * fill_light).sum(axis=1), 0.0, 1.0)
        shade = np.clip(0.52 + 0.38 * d_key + 0.16 * d_fill, 0.52, 1.06)
        shaded_rgb = np.clip(frgb * shade[:, None], 0, 255).astype(np.uint8)

        scale_px = tile_size * 0.41
        px = tri[:, :, 0] * scale_px + tile_size * 0.5
        py = -tri[:, :, 1] * scale_px + tile_size * 0.52
        tile = Image.new('RGB', (tile_size, tile_size), (22, 26, 34))
        draw = ImageDraw.Draw(tile)
        for pxs, pys, col in zip(px.tolist(), py.tolist(), shaded_rgb.tolist()):
            draw.polygon([(pxs[0], pys[0]), (pxs[1], pys[1]), (pxs[2], pys[2])], fill=tuple(col))
        strip.paste(tile, (idx * tile_size, 26))
        s_draw.text((idx * tile_size + 10, 6), title, fill=(220, 226, 236))
    strip.save(out_png)
    return out_png


def render_hf_trellis_modes(glb_path, out_png, tile_size=220):
    """Render HF TRELLIS Space-inspired 4-mode studio strip: Normal, Clay, Base Color, Studio HDRI."""
    verts, faces, face_rgb = _load_decimated_mesh_data(glb_path)
    tri, frgb, norms = _project_triangles(verts, faces, face_rgb, yaw_deg=30, pitch_deg=32)

    scale_px = tile_size * 0.41
    px = tri[:, :, 0] * scale_px + tile_size * 0.5
    py = -tri[:, :, 1] * scale_px + tile_size * 0.52

    key_l = np.array([0.40, 0.76, 0.51], dtype=np.float32)
    key_l /= np.linalg.norm(key_l)
    fill_l = np.array([-0.60, 0.30, 0.74], dtype=np.float32)
    fill_l /= np.linalg.norm(fill_l)
    rim_l = np.array([0.0, 0.45, -0.89], dtype=np.float32)
    d_key = np.clip((norms * key_l).sum(axis=1), 0.0, 1.0)
    d_fill = np.clip((norms * fill_l).sum(axis=1), 0.0, 1.0)
    d_rim = np.clip((norms * rim_l).sum(axis=1), 0.0, 1.0)

    normal_rgb = np.clip((norms * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
    clay_shade = np.clip(0.42 + 0.44 * d_key + 0.16 * d_fill + 0.14 * (d_rim ** 2), 0.42, 1.08)
    clay_rgb = np.clip(np.array([218.0, 206.0, 192.0], dtype=np.float32) * clay_shade[:, None], 0, 255).astype(np.uint8)
    base_rgb = np.clip(frgb, 0, 255).astype(np.uint8)
    hdri_tint = (
        0.54
        + 0.40 * d_key[:, None] * np.array([1.08, 0.99, 0.90], dtype=np.float32)
        + 0.18 * d_fill[:, None] * np.array([0.88, 0.96, 1.12], dtype=np.float32)
        + 0.12 * (d_rim[:, None] ** 2)
    )
    hdri_rgb = np.clip(frgb * np.clip(hdri_tint, 0.50, 1.15), 0, 255).astype(np.uint8)

    modes = [
        ('Normal', normal_rgb),
        ('Clay', clay_rgb),
        ('Base Color', base_rgb),
        ('Studio HDRI', hdri_rgb),
    ]
    strip = Image.new('RGB', (tile_size * len(modes), tile_size + 26), (20, 24, 32))
    s_draw = ImageDraw.Draw(strip)
    coords = [[(p0, q0), (p1, q1), (p2, q2)] for (p0, p1, p2), (q0, q1, q2) in zip(px.tolist(), py.tolist())]
    for m_idx, (title, colors) in enumerate(modes):
        tile = Image.new('RGB', (tile_size, tile_size), (18, 22, 30))
        t_draw = ImageDraw.Draw(tile)
        for poly, col in zip(coords, colors.tolist()):
            t_draw.polygon(poly, fill=tuple(col))
        strip.paste(tile, (m_idx * tile_size, 26))
        s_draw.text((m_idx * tile_size + 10, 6), title, fill=(224, 232, 242))
    strip.save(out_png)
    return out_png


extra_artifacts = []
for key, rec in RUNS_3D.items():
    if rec.get('status') == 'success' and rec.get('artifact_paths'):
        glb_path = Path(rec['artifact_paths'][0])
        preview_png = glb_path.with_name(glb_path.stem + '_preview.png')
        hf_modes_png = glb_path.with_name(glb_path.stem + '_hf_modes.png')
        try:
            render_glb_three_views(glb_path, preview_png)
            rec['artifact_paths'].append(preview_png)
            rec['preview_path'] = preview_png
            render_hf_trellis_modes(glb_path, hf_modes_png)
            rec['artifact_paths'].append(hf_modes_png)
            rec['hf_modes_path'] = hf_modes_png
        except Exception as err:
            print(f'Preview render failed for {key}:', type(err).__name__, err)

# Assemble comparison_sheet.png with Dish Input + Cutout (Plate + Food) + 3D backends
backends_order = [
    ('triposr', 'TripoSR (Vertex Color)'),
    ('hunyuan3d-2mini', 'Hunyuan3D-2mini (Geometry)'),
    ('trellis2-4b', 'TRELLIS.2-4B (PBR)'),
]
in_w = 240
cutout_w = 240
col_w = 540
row_h = 260
left_w = in_w + cutout_w
sheet_w = left_w + len(backends_order) * col_w
sheet_h = 56 + len(ACTIVE_LABELS) * row_h
sheet = Image.new('RGB', (sheet_w, sheet_h), (16, 20, 27))
draw = ImageDraw.Draw(sheet)
draw.text((16, 18), 'Dish Input (Source)', fill=(240, 244, 250))
draw.text((in_w + 16, 18), 'Cutout (Plate + Food)', fill=(240, 244, 250))
for c_idx, (_, title) in enumerate(backends_order):
    draw.text((left_w + c_idx * col_w + 14, 18), title, fill=(240, 244, 250))

for r_idx, label in enumerate(ACTIVE_LABELS):
    y0 = 56 + r_idx * row_h
    in_entry = image_results[label]
    draw.rectangle([8, y0 + 4, sheet_w - 8, y0 + row_h - 6], outline=(55, 65, 82), width=1)
    with Image.open(in_entry['image_path']) as im:
        contained = ImageOps.contain(im.convert('RGB'), (204, 200), Image.Resampling.LANCZOS)
        in_tile = Image.new('RGB', (204, 200), (22, 26, 34))
        in_tile.paste(contained, ((204 - contained.width) // 2, (200 - contained.height) // 2))
        sheet.paste(in_tile, (18, y0 + 38))
    src_tag = 'user' if in_entry['image_source'] == 'user-uploaded-dataset' else in_entry['image_source']
    draw.text((18, y0 + 12), f"{label} ({src_tag})", fill=(170, 210, 255))

    cut_path = in_entry.get('cutout_path')
    cm = in_entry.get('cutout_metrics', {})
    if cut_path and Path(cut_path).is_file():
        with Image.open(cut_path) as cim:
            contained_rgba = ImageOps.contain(cim.convert('RGBA'), (204, 200), Image.Resampling.LANCZOS)
            board = make_checkerboard((204, 200)).convert('RGBA')
            px_off = ((204 - contained_rgba.width) // 2, (200 - contained_rgba.height) // 2)
            board.alpha_composite(contained_rgba, px_off)
            sheet.paste(board.convert('RGB'), (in_w + 18, y0 + 38))
        pl_flag = 'plate=ON' if cm.get('plate_preserved') else 'plate=foodOnly'
        cut_cap = f"{pl_flag} (+{cm.get('plate_added_pct', 0)}%) {cm.get('fg_coverage_pct', '?')}%"
        draw.text((in_w + 18, y0 + 12), cut_cap[:34], fill=(170, 235, 205))

    for c_idx, (b_key, _) in enumerate(backends_order):
        x0 = left_w + c_idx * col_w
        rec = RUNS_3D.get(f'{label}/{b_key}', {})
        st = rec.get('status', 'not_run')
        if st == 'success' and rec.get('preview_path') and Path(rec['preview_path']).is_file():
            with Image.open(rec['preview_path']) as pv:
                pv_res = pv.resize((col_w - 20, 184), Image.Resampling.LANCZOS)
                sheet.paste(pv_res, (x0 + 10, y0 + 38))
            m = rec['metrics']
            mob_m = rec.get('mobile_metrics', {})
            v_num = m.get('vertices', m.get('vertex_count', 0))
            f_num = m.get('faces')
            f_txt = f'{f_num:,}' if isinstance(f_num, int) else '?'
            mob_txt = f" | mob={mob_m.get('faces', '?'):,}" if isinstance(mob_m.get('faces'), int) else ''
            cap = f"v={v_num:,} f={f_txt}{mob_txt} | {m.get('elapsed_seconds', '?')}s | {m.get('bytes', 0)//1024}KB"
            draw.text((x0 + 12, y0 + 12), cap[:72], fill=(175, 240, 185))
        else:
            msg = rec.get('error', st)[:62]
            draw.text((x0 + 14, y0 + 105), f"{st.upper()}: {msg}", fill=(245, 185, 130))

sheet_path = RESULTS_DIR / 'comparison_sheet.png'
sheet.save(sheet_path)
extra_artifacts.append(sheet_path)
display(DisplayImage(filename=str(sheet_path)))

# Assemble hf_modes_sheet.png (Normal / Clay / Base Color / Studio HDRI per dish & backend)
hf_backends = [('triposr', 'TripoSR - HF 4-Mode Strip'), ('hunyuan3d-2mini', 'Hunyuan3D-2mini - HF 4-Mode Strip')]
hf_label_w = 220
hf_col_w = 660
hf_row_h = 216
hf_w = hf_label_w + len(hf_backends) * hf_col_w
hf_h = 56 + len(ACTIVE_LABELS) * hf_row_h
hf_sheet = Image.new('RGB', (hf_w, hf_h), (14, 18, 25))
hf_draw = ImageDraw.Draw(hf_sheet)
hf_draw.text((16, 18), 'Dish + Preserved Cutout', fill=(240, 244, 250))
for c_idx, (_, title) in enumerate(hf_backends):
    hf_draw.text((hf_label_w + c_idx * hf_col_w + 14, 18), title, fill=(240, 244, 250))

for r_idx, label in enumerate(ACTIVE_LABELS):
    y0 = 56 + r_idx * hf_row_h
    hf_draw.rectangle([8, y0 + 4, hf_w - 8, y0 + hf_row_h - 6], outline=(55, 65, 82), width=1)
    cut_path = image_results[label].get('cutout_path')
    hf_draw.text((16, y0 + 10), label, fill=(170, 210, 255))
    if cut_path and Path(cut_path).is_file():
        with Image.open(cut_path) as cim:
            contained_rgba = ImageOps.contain(cim.convert('RGBA'), (184, 160), Image.Resampling.LANCZOS)
            board = make_checkerboard((184, 160)).convert('RGBA')
            board.alpha_composite(contained_rgba, ((184 - contained_rgba.width) // 2, (160 - contained_rgba.height) // 2))
            hf_sheet.paste(board.convert('RGB'), (16, y0 + 36))
    for c_idx, (b_key, _) in enumerate(hf_backends):
        x0 = hf_label_w + c_idx * hf_col_w
        rec = RUNS_3D.get(f'{label}/{b_key}', {})
        hm = rec.get('hf_modes_path')
        if rec.get('status') == 'success' and hm and Path(hm).is_file():
            with Image.open(hm) as hpv:
                hf_sheet.paste(hpv.resize((hf_col_w - 20, 176), Image.Resampling.LANCZOS), (x0 + 10, y0 + 28))

hf_sheet_path = RESULTS_DIR / 'hf_modes_sheet.png'
hf_sheet.save(hf_sheet_path)
extra_artifacts.append(hf_sheet_path)
display(DisplayImage(filename=str(hf_sheet_path)))

# Write per-dish worker_output.json receipts + Docker blueprint artifacts
for label in ACTIVE_LABELS:
    jm_path = Path(image_results[label]['job_manifest_path'])
    jm = json.loads(jm_path.read_text(encoding='utf-8'))
    dish_runs = {k: v for k, v in RUNS_3D.items() if v.get('input_label') == label}
    receipt = build_worker_receipt(jm, image_results[label], dish_runs, RESULTS_DIR)
    receipt_path = RESULTS_DIR / label / 'worker_output.json'
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
    extra_artifacts.append(receipt_path)

blueprint_dir = RESULTS_DIR / 'docker_blueprint'
blueprint_dir.mkdir(parents=True, exist_ok=True)
pinned_path = blueprint_dir / 'pinned_revisions.json'
pinned_path.write_text(json.dumps({
    'schema_version': 1,
    'demo_label': DEMO_TRUTHFUL_LABEL,
    'publication_eligible': False,
    'plate_preservation_mode': PLATE_PRESERVATION_MODE,
    'models': {k: v for k, v in REVISIONS.items() if not k.startswith(('input_', 'cutout_'))},
}, indent=2) + '\n', encoding='utf-8')
extra_artifacts.append(pinned_path)

lock_lines = [
    '# Pinned Python dependencies captured from the verified Kaggle CUDA runtime',
    *[f'{pkg}=={ver}' for pkg, ver in sorted(RUNTIME_PACKAGE_VERSIONS.items())],
]
lock_path = blueprint_dir / 'requirements.lock.txt'
lock_path.write_text('\n'.join(lock_lines) + '\n', encoding='utf-8')
extra_artifacts.append(lock_path)

worker_entrypoint_template = r"""# Docker worker entrypoint implementing the immutable job_manifest -> worker_output contract.

import argparse
import hashlib
import json
import os
from pathlib import Path


def sha256_file(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def verify_job_manifest(manifest_path, input_dir):
    manifest = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
    if manifest.get('publication_eligible') is not False:
        raise ValueError('Single-photo worker jobs must retain publication_eligible=false.')
    src_file = Path(input_dir) / Path(manifest['input_filename']).name
    if not src_file.is_file():
        raise FileNotFoundError(f'Missing input file: {src_file}')
    actual_sha = sha256_file(src_file)
    expected_sha = manifest.get('source_sha256')
    if not isinstance(expected_sha, str) or len(expected_sha) != 64 or any(c not in '0123456789abcdef' for c in expected_sha):
        raise ValueError('job_manifest.json must contain a valid 64-char lowercase hex source_sha256.')
    if actual_sha != expected_sha:
        raise ValueError(f'Source SHA-256 mismatch: expected {expected_sha}, got {actual_sha}')
    return manifest, src_file, actual_sha


def run_worker(manifest_path, input_dir, out_dir, dry_run=False):
    manifest, src_file, actual_sha = verify_job_manifest(manifest_path, input_dir)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    receipt = {
        'schema_version': 1,
        'job_id': manifest['job_id'],
        'dish_label': manifest['dish_label'],
        'demo_label': manifest.get('demo_label'),
        'publication_eligible': False,
        'source_sha256': actual_sha,
        'source_file': src_file.name,
        'plate_preservation_mode': manifest.get('plate_preservation_mode', 'preserve_plate'),
        'status': 'validated_dry_run' if dry_run else 'ready_for_gpu_backends',
        'export_profiles': manifest.get('export_profiles', {}),
        'pinned_revisions': manifest.get('pinned_revisions', {}),
    }
    receipt_path = out / 'worker_output.json'
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
    return receipt_path


def main():
    parser = argparse.ArgumentParser(description='PeekaPlate Docker 3D dish demo worker')
    parser.add_argument('--manifest', required=True, help='Path to immutable job_manifest.json')
    parser.add_argument('--input-dir', required=True, help='Directory holding the verified dish photo')
    parser.add_argument('--out', required=True, help='Output artifact directory')
    parser.add_argument('--dry-run', action='store_true', help='Verify contract & hashes without running GPU models')
    args = parser.parse_args()
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    out_receipt = run_worker(args.manifest, args.input_dir, args.out, dry_run=args.dry_run)
    print(f'Wrote verified receipt: {out_receipt}')


if __name__ == '__main__':
    main()
"""
entrypoint_path = blueprint_dir / 'worker_entrypoint.py'
entrypoint_path.write_text(worker_entrypoint_template, encoding='utf-8')
extra_artifacts.append(entrypoint_path)

_cuda_raw = (getattr(torch.version, 'cuda', None) or '12.8').strip()
_cuda_base = '.'.join(_cuda_raw.split('.')[:2]) if '.' in _cuda_raw else _cuda_raw
_cuda_tag = 'cu' + _cuda_base.replace('.', '')[:3]
dockerfile_template = f"""# PeekaPlate Dish-to-3D Demo Worker Template (CUDA {_cuda_base}, private ticket 86bd5m6q0 demo)
# Build context must provide requirements.lock.txt, pinned_revisions.json, and worker_entrypoint.py.
# Never bake HF_TOKEN, OPENROUTER_API_KEY, or customer photos into image layers.

FROM nvidia/cuda:{_cuda_base}.0-runtime-ubuntu24.04

ENV DEBIAN_FRONTEND=noninteractive \\
    PYTHONUNBUFFERED=1 \\
    PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \\
    HF_HOME=/var/cache/peekaplate/hf \\
    TORCH_HOME=/var/cache/peekaplate/torch \\
    U2NET_HOME=/var/cache/peekaplate/u2net

RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3 python3-pip git libgl1 libglib2.0-0 libgomp1 \\
    && rm -rf /var/lib/apt/lists/*

RUN useradd -m -u 10001 worker \\
    && mkdir -p /app /var/cache/peekaplate /work \\
    && chown -R worker:worker /app /var/cache/peekaplate /work

WORKDIR /app
COPY requirements.lock.txt /app/requirements.lock.txt
RUN pip3 install --no-cache-dir --break-system-packages \\
    --extra-index-url https://download.pytorch.org/whl/{_cuda_tag} \\
    -r /app/requirements.lock.txt

COPY pinned_revisions.json /app/pinned_revisions.json
COPY worker_entrypoint.py /app/worker_entrypoint.py

USER worker
VOLUME ["/var/cache/peekaplate", "/work"]
ENTRYPOINT ["python3", "/app/worker_entrypoint.py"]
CMD ["--help"]
"""
dockerfile_path = blueprint_dir / 'Dockerfile.worker.template'
dockerfile_path.write_text(dockerfile_template, encoding='utf-8')
extra_artifacts.append(dockerfile_path)

# Interactive HF TRELLIS Space-inspired studio viewer HTML
dish_panels = []
for idx, label in enumerate(ACTIVE_LABELS):
    in_entry = image_results[label]
    cm = in_entry.get('cutout_metrics', {})
    rel_in = Path(in_entry['image_path']).relative_to(RESULTS_DIR).as_posix()
    rel_cut = Path(in_entry['cutout_path']).relative_to(RESULTS_DIR).as_posix()
    cards = []
    for key, rec in RUNS_3D.items():
        if rec.get('input_label') != label or rec.get('status') != 'success':
            continue
        rel_raw = Path(rec['artifact_paths'][0]).relative_to(RESULTS_DIR).as_posix()
        rel_mob = Path(rec['mobile_glb_path']).relative_to(RESULTS_DIR).as_posix() if rec.get('mobile_glb_path') else rel_raw
        rel_hf = Path(rec['hf_modes_path']).relative_to(RESULTS_DIR).as_posix() if rec.get('hf_modes_path') else ''
        m = rec['metrics']
        mob_m = rec.get('mobile_metrics', {})
        mv_id = f'mv_{label}_{rec["backend"].replace("-", "_").replace(".", "_")}'
        hf_img = f"<img class='hf-strip' src='{html_lib.escape(rel_hf)}' alt='HF 4 modes'>" if rel_hf else ''
        cards.append(
            f"<div class='card'>"
            f"<div class='card-top'><h3>{html_lib.escape(rec['backend'])}</h3>"
            f"<div class='seg-btns'>"
            f"<button class='btn active' onclick=\"setModelSrc('{html_lib.escape(mv_id)}', '{html_lib.escape(rel_raw)}', this)\">Raw GLB ({m.get('faces', '?')}f)</button>"
            f"<button class='btn' onclick=\"setModelSrc('{html_lib.escape(mv_id)}', '{html_lib.escape(rel_mob)}', this)\">Mobile GLB ({mob_m.get('faces', '?')}f)</button>"
            f"<a class='btn' href='{html_lib.escape(rel_mob)}' download>Download</a>"
            f"</div></div>"
            f"<p class='stats'>appearance={html_lib.escape(rec['appearance'])} | raw={m.get('bytes', 0)//1024}KB | mobile={mob_m.get('bytes', 0)//1024}KB | {m.get('elapsed_seconds', '?')}s</p>"
            f"<model-viewer id='{html_lib.escape(mv_id)}' src='{html_lib.escape(rel_raw)}' camera-controls auto-rotate exposure='1.1' shadow-intensity='1'></model-viewer>"
            f"{hf_img}"
            f"</div>"
        )
    active_cls = ' active' if idx == 0 else ''
    dish_panels.append(
        f"<section class='dish-panel{active_cls}' data-dish='{html_lib.escape(label)}'>"
        f"<div class='dish-inputs'>"
        f"<div class='in-box'><span>Source Photo</span><img src='{html_lib.escape(rel_in)}'></div>"
        f"<div class='in-box'><span>Plate + Food Cutout ({'PRESERVED' if cm.get('plate_preserved') else 'food-only'} | +{cm.get('plate_added_pct', 0)}% plate)</span><img class='checker' src='{html_lib.escape(rel_cut)}'></div>"
        f"</div>"
        f"<div class='grid'>{''.join(cards)}</div>"
        f"</section>"
    )

tab_buttons = ''.join(
    f"<button class='tab-btn{' active' if i == 0 else ''}' onclick=\"selectDish('{html_lib.escape(lbl)}', this)\">{html_lib.escape(lbl)}</button>"
    for i, lbl in enumerate(ACTIVE_LABELS)
)

viewer_html = (
    "<!doctype html><html lang='en'><head><meta charset='utf-8'>"
    "<title>PeekaPlate — Plate-Preserving 3D Dish Demo</title>"
    "<script type='module' src='https://unpkg.com/@google/model-viewer@4.0.0/dist/model-viewer.min.js'></script>"
    "<style>"
    "body{font:14px system-ui,sans-serif;background:#0d1118;color:#eef2f7;margin:0;padding:20px}"
    ".hdr{display:flex;flex-wrap:wrap;justify-content:space-between;align-items:center;gap:12px;margin-bottom:14px}"
    ".badge{background:#233146;color:#93c5fd;border:1px solid #3b5375;padding:4px 10px;border-radius:999px;font-size:12px}"
    ".controls{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-bottom:16px}"
    ".tabs{display:flex;flex-wrap:wrap;gap:8px}"
    ".tab-btn,.btn{background:#18202c;color:#dbe7f5;border:1px solid #2e3b4f;border-radius:6px;padding:6px 12px;cursor:pointer;text-decoration:none;font-size:13px}"
    ".tab-btn.active,.btn.active{background:#2563eb;border-color:#3b82f6;color:#fff}"
    ".dish-panel{display:none}.dish-panel.active{display:block}"
    ".dish-inputs{display:flex;flex-wrap:wrap;gap:12px;margin-bottom:16px}"
    ".in-box{background:#151c27;border:1px solid #273244;border-radius:8px;padding:10px;display:flex;flex-direction:column;gap:6px}"
    ".in-box img{height:180px;max-width:300px;object-fit:contain;border-radius:4px}"
    ".checker{background:repeating-conic-gradient(#1f2733 0% 25%, #2b3545 0% 50%) 50% / 20px 20px}"
    ".grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:16px}"
    ".card{background:#161d28;border:1px solid #2c384b;border-radius:10px;padding:12px}"
    ".card-top{display:flex;justify-content:space-between;align-items:center;gap:8px;flex-wrap:wrap}"
    ".seg-btns{display:flex;gap:6px;flex-wrap:wrap}.stats{color:#9db0ca;margin:6px 0}"
    "model-viewer{width:100%;height:360px;background:#090c12;border-radius:6px}"
    ".hf-strip{width:100%;margin-top:8px;border-radius:6px;border:1px solid #283244}"
    "</style></head><body>"
    "<div class='hdr'><h1>PeekaPlate — Plate-Preserving 3D Dish Demo</h1>"
    f"<span class='badge'>{html_lib.escape(DEMO_TRUTHFUL_LABEL)} | publication_eligible=false</span></div>"
    "<div class='controls'>"
    f"<div class='tabs'>{tab_buttons}</div>"
    "<button class='btn' onclick='toggleRotate()'>Toggle Auto-Rotate</button>"
    "<button class='btn' onclick='resetCameras()'>Reset Cameras</button>"
    "<label>Exposure <input type='range' min='0.6' max='1.8' step='0.1' value='1.1' oninput='setExposure(this.value)'></label>"
    "</div>"
    + ''.join(dish_panels)
    + "<script>"
    "function selectDish(name,btn){document.querySelectorAll('.dish-panel').forEach(p=>p.classList.toggle('active',p.dataset.dish===name));"
    "document.querySelectorAll('.tab-btn').forEach(b=>b.classList.remove('active'));btn.classList.add('active');}"
    "function setModelSrc(id,src,btn){const mv=document.getElementById(id);if(mv)mv.src=src;"
    "btn.parentElement.querySelectorAll('button').forEach(b=>b.classList.remove('active'));btn.classList.add('active');}"
    "function toggleRotate(){document.querySelectorAll('model-viewer').forEach(mv=>mv.autoRotate=!mv.autoRotate);}"
    "function resetCameras(){document.querySelectorAll('model-viewer').forEach(mv=>{mv.cameraOrbit='auto auto auto';if(mv.resetTurntableRotation)mv.resetTurntableRotation();});}"
    "function setExposure(v){document.querySelectorAll('model-viewer').forEach(mv=>mv.exposure=parseFloat(v));}"
    "</script></body></html>"
)
viewer_path = RESULTS_DIR / 'comparison_viewer.html'
viewer_path.write_text(viewer_html, encoding='utf-8')
extra_artifacts.append(viewer_path)

manifest_path, zip_path = write_results(RESULTS_DIR, image_results, RUNS_3D, SETTINGS, REVISIONS, extra_artifacts)
display(FileLink(str(manifest_path)))
display(FileLink(str(zip_path)))

print('\n=== FINAL PLATE-PRESERVING DEMO SUMMARY ===')
for key, rec in RUNS_3D.items():
    m = rec.get('metrics', {})
    mob = rec.get('mobile_metrics', {})
    print(f"{key:36s} | {rec['status']:7s} | {rec['appearance']:14s} | "
          f"raw_f={str(m.get('faces', '-')):>7s} | mob_f={str(mob.get('faces', '-')):>6s} | "
          f"raw_B={str(m.get('bytes', '-')):>9s} | mob_B={str(mob.get('bytes', '-')):>8s}")
